# DCC 2026 Mission 2 — Baseline Freeze + Baseline Error Analysis

이 Notebook은 지연우의 **MFCC40 + Logistic Regression** baseline을 먼저 재현·동결한 뒤,
공식 Validation의 모든 prediction을 원본 발화와 연결해 오류 패턴을 분석하기 위한 실행 틀이다.

**이 파일에는 실행 결과가 저장되어 있지 않다.** Codex는 실제 데이터, MFCC cache, 모델 학습,
Validation prediction, Error Analysis, 음성 청취를 실행하지 않았다. 아래의 무거운 단계는 기본적으로
`False`인 명시적 switch와 앞 checkpoint gate로 차단되어 있다.


## Goal과 규정 경계

공식/프로젝트 근거의 우선순위는 저장소의 `AGENTS.md`, `docs/dcc_rules.md`, 프로젝트 reference의
최신 공식 문제·관리자 Q&A/공지 통합본을 따른다. Mission 2의 입력은 `startAt`/`endAt`으로 자른
**한 사람의 한 발화 음성**이다. `speaker=0`은 119대원, `speaker=1`은 신고자이며 공식 metric은 Accuracy다.

모델 입력 `X`에는 waveform에서 계산한 MFCC mean/std 40개만 들어간다. `text`, 다른 annotation,
`call_id`, `utterance_id`, `utt_idx`, 절대 `startAt`/`endAt`, overlap, quality flag는 모델 feature가 아니다.
이 값들은 prediction 추적과 Error Analysis에만 사용한다.

분석 모집단을 섞지 않는다.

- duration: 공식 Validation 전체 **111,947행**
- acoustic / silence / RMS / quality: 기존 **1,200개 stratified sample 중 Validation 600행**
- duplicate conflict: **Training 전용 3 groups / 6 rows 별도 audit**
- overlap: Validation 발화의 분석/추적 metadata


## 실행 순서와 STOP 원칙

1. `ea-imports` → `ea-config` → 다섯 helper cell → `ea-synthetic-checks`
2. Checkpoint 1 source/cache 계약 확인
3. 기본값 `RUN_BASELINE_REPRODUCTION=True`로 baseline 재학습·예측·동결 gate
4. Checkpoint 2 원본 JSON 순서 및 1:1 ledger join gate
5. 기본값 `RUN_ERROR_ANALYSIS=True`로 Checkpoint 3 Error Analysis
6. 기본값 `RUN_LISTENING_SELECTION=True`로 blind listening 후보 선정
7. 청취할 때 사용자가 `ENABLE_AUDIO_PLAYBACK=True`로 직접 바꾼 뒤 재생
8. 모든 청취 note 작성 후 사용자가 `REVEAL_LISTENING_ANSWERS=True`로 직접 바꾼 뒤 정답 공개

`Restart & Run All`은 분석과 청취 후보 선정까지 자동 진행할 수 있다. playback은 기본값 `False`에서
재생 전에 STOP하며, answer reveal도 사용자가 직접 켜기 전에는 실행되지 않는다.
assertion 또는 `RuntimeError`가 발생하면 그 checkpoint에서 멈추고 원인을 확인한다.


In [1]:
from __future__ import annotations

import hashlib
import json
import sys
import warnings
from collections import Counter
from numbers import Integral, Real
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Audio, display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score
from sklearn.preprocessing import StandardScaler


In [2]:
# [먼저 실행] 경로와 실행 switch만 정의한다. 실제 데이터/모델은 아직 읽거나 실행하지 않는다.
current = Path.cwd().resolve()
if (current / "src").is_dir():
    REPO_ROOT = current
elif (current.parent / "src").is_dir():
    REPO_ROOT = current.parent
else:
    raise FileNotFoundError(f"Repository root를 찾지 못했습니다: {current}")

DATA_ROOT = Path(
    __import__("os").environ.get(
        "DCC_DATA_ROOT", REPO_ROOT / "data" / "raw_original"
    )
).expanduser().resolve()
BASELINE_NOTEBOOK_PATH = REPO_ROOT / "notebooks" / "mission2_yeonwoo_baseline.ipynb"
BASELINE_CACHE_ROOT = REPO_ROOT / "data" / "processed" / "mission2"
EDA_CACHE_ROOT = REPO_ROOT / "tmp" / "mission2_eda"
EDA_UTTERANCE_PATH = EDA_CACHE_ROOT / "utterance_manifest_v1.parquet"
EDA_ACOUSTIC_PATH = EDA_CACHE_ROOT / "acoustic_features_sample_v1.parquet"
EDA_DUPLICATE_PATH = EDA_CACHE_ROOT / "duplicate_intervals_v1.parquet"

EXPECTED_BASELINE_CODE_SHA256 = "8a628a44801a3925360f91adf4b03b3f3e7dbdd9eb7ec20ccf0670340d6ecd13"
EXPECTED_CALL_COUNTS = {"Training": 29_200, "Validation": 3_640}
EXPECTED_UTTERANCE_COUNTS = {"Training": 889_358, "Validation": 111_947}
MFCC_CONFIG = {"sample_rate": 8_000, "n_mfcc": 20, "n_fft": 512, "hop_length": 160}
FEATURE_DIMENSION = 40
MODEL_FEATURE_NAMES = tuple(
    [f"mfcc_{index:02d}_mean" for index in range(1, 21)]
    + [f"mfcc_{index:02d}_std" for index in range(1, 21)]
)
FORBIDDEN_MODEL_FIELDS = {
    "text", "speaker", "call_id", "record_key", "utterance_id", "utt_idx",
    "annotation_row", "startAt", "endAt", "start_at", "end_at", "duration",
    "overlap_status", "quality_status",
}

# 저장된 baseline 결과에서 직접 확인한 exact reference matrix다.
# Accuracy/Recall 숫자는 별도로 하드코딩하지 않고 이 matrix에서 계산한다.
REFERENCE_CONFUSION_MATRIX = np.array([[41_190, 12_655], [8_715, 49_387]], dtype=np.int64)

RUN_BASELINE_REPRODUCTION = True
RUN_ERROR_ANALYSIS = True
RUN_LISTENING_SELECTION = True
ENABLE_AUDIO_PLAYBACK = False
REVEAL_LISTENING_ANSWERS = False
SAVE_ANALYSIS_ARTIFACTS = False

HIGH_CONFIDENCE_THRESHOLD = 0.90  # 분석/청취 후보용; prediction threshold를 바꾸지 않는다.
LOW_CONFIDENCE_THRESHOLD = 0.60   # 분석/청취 후보용; prediction threshold를 바꾸지 않는다.
LISTENING_PER_GROUP = 3
LISTENING_MAX_PER_CALL = 1

print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("Setup only: no competition data, model, or prediction has run.")


REPO_ROOT: C:\Users\YW\Study\02_Projects\DCC_2026\prelim
DATA_ROOT: C:\Users\YW\Study\02_Projects\DCC_2026\prelim\data\raw_original
Setup only: no competition data, model, or prediction has run.


In [13]:
def require_columns(frame: pd.DataFrame, columns: list[str] | tuple[str, ...], name: str) -> None:
    missing = sorted(set(columns) - set(frame.columns))
    if missing:
        raise AssertionError(f"{name} missing columns: {missing}")


def make_record_key(split: object, call_id: object, annotation_row: object) -> str:
    """Return split|call_id|zero-based original JSON-array position."""
    split_text = str(split)
    call_text = str(call_id)
    if "|" in split_text or "|" in call_text:
        raise AssertionError("record_key components must not contain '|'")
    if not isinstance(annotation_row, Integral) or isinstance(annotation_row, (bool, np.bool_)):
        raise AssertionError(f"annotation_row must be an integer: {annotation_row!r}")
    position = int(annotation_row)
    if position < 0:
        raise AssertionError(f"annotation_row must be non-negative: {position}")
    return f"{split_text}|{call_text}|{position}"


def rebuild_original_json_order(
    validation_manifest: pd.DataFrame,
    data_root: Path,
    *,
    expected_rows: int,
) -> pd.DataFrame:
    """Re-read only allowed tracking fields in original JSON-array order.

    JSON arrays are ordered. enumerate(utterances) therefore reproduces the same
    zero-based annotation_row as long as the official JSON file is unchanged.
    `text` and all unrelated annotations are deliberately never copied.
    """
    require_columns(
        validation_manifest,
        ["split", "call_id", "json_path"],
        "validation_manifest",
    )
    call_sources = (
        validation_manifest[["split", "call_id", "json_path"]]
        .drop_duplicates()
        .sort_values(["call_id", "json_path"], kind="stable")
        .reset_index(drop=True)
    )
    assert set(call_sources["split"]) == {"Validation"}, "Only Validation JSON is allowed here"
    assert call_sources["call_id"].is_unique, "A Validation call has multiple JSON paths"

    rows: list[dict[str, object]] = []
    for source in call_sources.itertuples(index=False):
        json_path = data_root / source.json_path
        with json_path.open("r", encoding="utf-8") as stream:
            document = json.load(stream)
        utterances = document.get("utterances")
        assert isinstance(utterances, list), f"utterances is not a list: {json_path}"
        for annotation_row, utterance in enumerate(utterances):
            assert isinstance(utterance, dict), f"utterance is not an object: {json_path}"
            start_at = utterance.get("startAt")
            end_at = utterance.get("endAt")
            speaker = utterance.get("speaker")
            assert isinstance(start_at, Real) and not isinstance(start_at, bool)
            assert isinstance(end_at, Real) and not isinstance(end_at, bool)
            assert isinstance(speaker, Integral) and not isinstance(speaker, (bool, np.bool_))
            assert int(speaker) in (0, 1)
            rows.append(
                {
                    "split": "Validation",
                    "call_id": str(source.call_id),
                    "annotation_row": int(annotation_row),
                    "record_key": make_record_key("Validation", source.call_id, annotation_row),
                    "utterance_id": utterance.get("id"),
                    "startAt_ms": float(start_at),
                    "endAt_ms": float(end_at),
                    "speaker": int(speaker),
                    "json_path": str(source.json_path),
                }
            )

    rebuilt = pd.DataFrame(rows)
    assert len(rebuilt) == expected_rows, (len(rebuilt), expected_rows)
    assert rebuilt["record_key"].is_unique, "Original-JSON record_key is not unique"
    for (_, _), group in rebuilt.groupby(["split", "call_id"], sort=False):
        expected_positions = np.arange(len(group), dtype=np.int64)
        np.testing.assert_array_equal(group["annotation_row"].to_numpy(), expected_positions)
    return rebuilt


def validate_eda_manifest_against_source(
    source_order: pd.DataFrame,
    eda_validation: pd.DataFrame,
    *,
    expected_rows: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Require an exact one-to-one match between original JSON order and EDA manifest."""
    required = [
        "record_key", "split", "call_id", "annotation_row", "utterance_id",
        "startAt_ms", "endAt_ms", "speaker", "json_path",
    ]
    require_columns(source_order, required, "source_order")
    require_columns(eda_validation, required, "eda_validation")
    source_duplicates = int(source_order["record_key"].duplicated(keep=False).sum())
    eda_duplicates = int(eda_validation["record_key"].duplicated(keep=False).sum())
    assert source_duplicates == 0, f"Original JSON duplicate keys: {source_duplicates}"
    assert eda_duplicates == 0, f"EDA manifest duplicate keys: {eda_duplicates}"
    assert len(source_order) == expected_rows
    assert len(eda_validation) == expected_rows

    merged = source_order.merge(
        eda_validation,
        on="record_key",
        how="outer",
        suffixes=("_source", "_eda"),
        indicator=True,
        validate="one_to_one",
    )
    missing = int((merged["_merge"] != "both").sum())
    assert len(merged) == expected_rows, (len(merged), expected_rows)
    assert missing == 0, f"Original JSON ↔ EDA missing rows: {missing}"
    for column in ["split", "call_id", "annotation_row", "speaker", "json_path"]:
        assert (merged[f"{column}_source"].astype(str) == merged[f"{column}_eda"].astype(str)).all(), column
    for column in ["startAt_ms", "endAt_ms"]:
        np.testing.assert_allclose(
            merged[f"{column}_source"].to_numpy(dtype=float),
            merged[f"{column}_eda"].to_numpy(dtype=float),
            rtol=0,
            atol=0,
        )

    audit = pd.DataFrame(
        [{
            "expected_rows": expected_rows,
            "source_rows": len(source_order),
            "eda_rows": len(eda_validation),
            "joined_rows": len(merged),
            "source_duplicate_keys": source_duplicates,
            "eda_duplicate_keys": eda_duplicates,
            "missing_join_rows": missing,
        }]
    )
    return merged, audit


def join_baseline_cache_to_manifest(
    validation_bundle: dict[str, np.ndarray],
    eda_validation: pd.DataFrame,
    *,
    expected_rows: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Attach original record_key by cached row order within each call, then verify all fields."""
    required_arrays = ["stem", "start_at", "end_at", "duration", "y"]
    missing_arrays = sorted(set(required_arrays) - set(validation_bundle))
    assert not missing_arrays, f"Validation cache missing arrays: {missing_arrays}"
    lengths = {name: len(validation_bundle[name]) for name in required_arrays}
    assert len(set(lengths.values())) == 1, f"Validation cache length mismatch: {lengths}"

    baseline = pd.DataFrame(
        {
            "baseline_row_idx": np.arange(lengths["y"], dtype=np.int64),
            "call_id_cache": np.asarray(validation_bundle["stem"]).astype(str),
            "startAt_cache": np.asarray(validation_bundle["start_at"], dtype=float),
            "endAt_cache": np.asarray(validation_bundle["end_at"], dtype=float),
            "duration_ms_cache": np.asarray(validation_bundle["duration"], dtype=float),
            "y_cache": np.asarray(validation_bundle["y"], dtype=np.int64),
        }
    )
    baseline["utt_idx"] = baseline.groupby("call_id_cache", sort=False).cumcount()
    baseline["record_key"] = [
        make_record_key("Validation", call_id, int(utt_idx))
        for call_id, utt_idx in zip(baseline["call_id_cache"], baseline["utt_idx"])
    ]

    baseline_duplicate_keys = int(baseline["record_key"].duplicated(keep=False).sum())
    eda_duplicate_keys = int(eda_validation["record_key"].duplicated(keep=False).sum())
    assert len(baseline) == expected_rows, (len(baseline), expected_rows)
    assert len(eda_validation) == expected_rows, (len(eda_validation), expected_rows)
    assert baseline_duplicate_keys == 0, f"Baseline duplicate keys: {baseline_duplicate_keys}"
    assert eda_duplicate_keys == 0, f"EDA duplicate keys: {eda_duplicate_keys}"

    joined = baseline.merge(
        eda_validation,
        on="record_key",
        how="outer",
        indicator=True,
        validate="one_to_one",
    )
    missing_join_rows = int((joined["_merge"] != "both").sum())
    assert len(joined) == expected_rows, (len(joined), expected_rows)
    assert missing_join_rows == 0, f"Baseline cache ↔ EDA missing rows: {missing_join_rows}"
    joined = joined.sort_values("baseline_row_idx", kind="stable").reset_index(drop=True)

    assert (joined["call_id_cache"] == joined["call_id"].astype(str)).all()
    np.testing.assert_array_equal(
        joined["utt_idx"].to_numpy(dtype=np.int64),
        joined["annotation_row"].to_numpy(dtype=np.int64),
    )
    np.testing.assert_allclose(joined["startAt_cache"], joined["startAt_ms"], rtol=0, atol=0)
    np.testing.assert_allclose(joined["endAt_cache"], joined["endAt_ms"], rtol=0, atol=0)
    np.testing.assert_allclose(
        joined["duration_ms_cache"],
        joined["endAt_ms"] - joined["startAt_ms"],
        rtol=0,
        atol=0,
    )
    np.testing.assert_array_equal(
        joined["y_cache"].to_numpy(dtype=np.int64),
        joined["speaker"].to_numpy(dtype=np.int64),
    )

    audit = pd.DataFrame(
        [{
            "expected_rows": expected_rows,
            "baseline_cache_rows": len(baseline),
            "eda_manifest_rows": len(eda_validation),
            "joined_rows": len(joined),
            "baseline_duplicate_keys": baseline_duplicate_keys,
            "eda_duplicate_keys": eda_duplicate_keys,
            "duplicate_join_keys": baseline_duplicate_keys + eda_duplicate_keys,
            "missing_join_rows": missing_join_rows,
        }]
    )
    return joined, audit


In [10]:
def metrics_from_confusion_matrix(matrix: np.ndarray) -> dict[str, float | int]:
    matrix = np.asarray(matrix, dtype=np.int64)
    assert matrix.shape == (2, 2)
    tn, fp, fn, tp = matrix.ravel()
    total = int(matrix.sum())
    support_0 = int(tn + fp)
    support_1 = int(fn + tp)
    return {
        "sample_count": total,
        "correct_count": int(tn + tp),
        "accuracy": (tn + tp) / total if total else np.nan,
        "speaker_0_recall": tn / support_0 if support_0 else np.nan,
        "speaker_1_recall": tp / support_1 if support_1 else np.nan,
        "fp_0_to_1": int(fp),
        "fn_1_to_0": int(fn),
        "fp_rate_within_true_0": fp / support_0 if support_0 else np.nan,
        "fn_rate_within_true_1": fn / support_1 if support_1 else np.nan,
        "support_0": support_0,
        "support_1": support_1,
    }


def evaluate_baseline_gate(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    *,
    expected_rows: int,
    reference_matrix: np.ndarray,
) -> tuple[np.ndarray, pd.DataFrame]:
    """Exact-gate row count and confusion matrix; derive all rates from that matrix."""
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    assert len(y_true) == expected_rows, (len(y_true), expected_rows)
    assert y_true.shape == y_pred.shape
    observed_matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
    np.testing.assert_array_equal(observed_matrix, np.asarray(reference_matrix, dtype=np.int64))

    derived = metrics_from_confusion_matrix(observed_matrix)
    sklearn_accuracy = float(accuracy_score(y_true, y_pred))
    sklearn_recalls = recall_score(
        y_true, y_pred, labels=[0, 1], average=None, zero_division=0
    )
    assert np.isclose(sklearn_accuracy, derived["accuracy"], rtol=0, atol=1e-15)
    np.testing.assert_allclose(
        sklearn_recalls,
        [derived["speaker_0_recall"], derived["speaker_1_recall"]],
        rtol=0,
        atol=1e-15,
    )
    return observed_matrix, pd.DataFrame([derived])


def error_type_values(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    assert y_true.shape == y_pred.shape
    return np.select(
        [
            (y_true == 0) & (y_pred == 0),
            (y_true == 1) & (y_pred == 1),
            (y_true == 0) & (y_pred == 1),
            (y_true == 1) & (y_pred == 0),
        ],
        ["correct_0", "correct_1", "0_to_1", "1_to_0"],
        default="invalid",
    )


def build_prediction_ledger(
    joined_tracking: pd.DataFrame,
    y_pred: np.ndarray,
    probabilities: np.ndarray,
    model_classes: np.ndarray,
) -> pd.DataFrame:
    joined = joined_tracking.sort_values("baseline_row_idx", kind="stable").reset_index(drop=True)
    y_true = joined["y_cache"].to_numpy(dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    probabilities = np.asarray(probabilities, dtype=float)
    classes = np.asarray(model_classes, dtype=np.int64)
    np.testing.assert_array_equal(classes, np.array([0, 1], dtype=np.int64))
    assert probabilities.shape == (len(joined), 2)
    assert y_pred.shape == y_true.shape
    assert np.isfinite(probabilities).all()
    np.testing.assert_allclose(probabilities.sum(axis=1), 1.0, rtol=0, atol=1e-7)

    ledger = pd.DataFrame(
        {
            "record_key": joined["record_key"].astype(str),
            "call_id": joined["call_id"].astype(str),
            "utterance_id": joined["utterance_id"],
            "utt_idx": joined["annotation_row"].astype(np.int64),
            "startAt": joined["startAt_ms"].astype(float),
            "endAt": joined["endAt_ms"].astype(float),
            "duration": (joined["endAt_ms"] - joined["startAt_ms"]) / 1000.0,
            "y_true": y_true,
            "y_pred": y_pred,
            "p0": probabilities[:, 0],
            "p1": probabilities[:, 1],
            "confidence": probabilities.max(axis=1),
            "correct": y_true == y_pred,
            "error_type": error_type_values(y_true, y_pred),
            "audio_path": joined["audio_path"].astype(str),
            "json_path": joined["json_path"].astype(str),
            "crop_ok": joined["crop_ok"].astype(bool),
            "boundary_status": joined["boundary_status"].astype(str),
        }
    )
    assert ledger["record_key"].is_unique
    assert set(ledger["error_type"]) <= {"correct_0", "correct_1", "0_to_1", "1_to_0"}
    return ledger


def summarize_binary_frame(
    frame: pd.DataFrame,
    slice_column: str | None = None,
) -> pd.DataFrame:
    require_columns(frame, ["y_true", "y_pred", "confidence"], "metric frame")
    groups = [("all", frame)] if slice_column is None else frame.groupby(
        slice_column, observed=True, dropna=False
    )
    rows: list[dict[str, object]] = []
    for slice_value, group in groups:
        y_true = group["y_true"].to_numpy(dtype=np.int64)
        y_pred = group["y_pred"].to_numpy(dtype=np.int64)
        matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
        metrics = metrics_from_confusion_matrix(matrix)
        metrics["slice"] = str(slice_value)
        metrics["mean_confidence"] = float(group["confidence"].mean()) if len(group) else np.nan
        error_confidence = group.loc[y_true != y_pred, "confidence"]
        metrics["mean_error_confidence"] = (
            float(error_confidence.mean()) if len(error_confidence) else np.nan
        )
        rows.append(metrics)
    columns = ["slice", "sample_count", "accuracy", "speaker_0_recall", "speaker_1_recall",
               "fp_0_to_1", "fn_1_to_0", "fp_rate_within_true_0", "fn_rate_within_true_1",
               "support_0", "support_1", "mean_confidence", "mean_error_confidence", "correct_count"]
    return pd.DataFrame(rows)[columns]


In [18]:
def add_overlap_metadata(frame: pd.DataFrame) -> pd.DataFrame:
    """Add utterance-level overlap metadata; never return it as a model feature."""
    require_columns(frame, ["record_key", "call_id", "startAt", "endAt"], "overlap frame")
    result = frame.reset_index(drop=True).copy()
    partner_count = np.zeros(len(result), dtype=np.int64)
    maximum_ratio = np.zeros(len(result), dtype=float)

    for _, group in result.groupby("call_id", sort=False):
        ordered_positions = sorted(
            group.index.tolist(),
            key=lambda position: (
                float(result.at[position, "startAt"]),
                float(result.at[position, "endAt"]),
                str(result.at[position, "record_key"]),
            ),
        )
        for left_offset, left_position in enumerate(ordered_positions):
            left_start = float(result.at[left_position, "startAt"])
            left_end = float(result.at[left_position, "endAt"])
            assert np.isfinite(left_start) and np.isfinite(left_end) and left_end > left_start
            for right_position in ordered_positions[left_offset + 1 :]:
                right_start = float(result.at[right_position, "startAt"])
                if right_start >= left_end:  # half-open [startAt, endAt): touching is not overlap
                    break
                right_end = float(result.at[right_position, "endAt"])
                overlap = min(left_end, right_end) - max(left_start, right_start)
                if overlap <= 0:
                    continue
                shorter = min(left_end - left_start, right_end - right_start)
                ratio = overlap / shorter
                partner_count[left_position] += 1
                partner_count[right_position] += 1
                maximum_ratio[left_position] = max(maximum_ratio[left_position], ratio)
                maximum_ratio[right_position] = max(maximum_ratio[right_position], ratio)

    result["overlap_partner_count"] = partner_count
    result["maximum_overlap_ratio"] = maximum_ratio
    result["overlap_status"] = np.where(partner_count > 0, "overlap", "no_overlap")
    return result


In [20]:
CORE_ACOUSTIC_FEATURES = [
    "duration_sec", "rms", "rms_frame_std", "peak_amplitude",
    "amplitude_dynamic_range", "silence_ratio", "zcr", "spectral_centroid",
    "spectral_bandwidth", "spectral_rolloff", "spectral_flatness", "clipping_ratio",
]


def weighted_quantile(values, weights, quantiles) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    quantiles = np.asarray(quantiles, dtype=float)
    keep = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values, weights = values[keep], weights[keep]
    if len(values) == 0:
        return np.full(len(quantiles), np.nan)
    order = np.argsort(values)
    values, weights = values[order], weights[order]
    cumulative = np.cumsum(weights) - 0.5 * weights
    cumulative = cumulative / weights.sum()
    return np.interp(quantiles, cumulative, values)


def adaptive_three_level_bins(
    low_cut: float,
    high_cut: float,
    labels: list[str],
    dimension: str,
) -> tuple[list[float], list[str]]:
    cutoffs = sorted(set(float(value) for value in [low_cut, high_cut] if np.isfinite(value)))
    if len(cutoffs) == 2:
        return [-np.inf, *cutoffs, np.inf], labels
    if len(cutoffs) == 1:
        warnings.warn(f"{dimension} cutoff ties: collapsing from three slices to two")
        return [-np.inf, cutoffs[0], np.inf], [f"{labels[0]}_or_equal", labels[-1]]
    raise AssertionError(f"No valid {dimension} cutoffs")


def build_acoustic_slice_frame(
    acoustic_features: pd.DataFrame,
    full_manifest: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Reproduce Raw EDA Step 9/12 labels from saved features; no audio is re-extracted."""
    require_columns(
        acoustic_features,
        ["record_key", "split", "speaker", "sample_weight", "extraction_ok", *CORE_ACOUSTIC_FEATURES],
        "acoustic_features",
    )
    require_columns(
        full_manifest,
        ["record_key", "split", "speaker", "duration_sec", "channels", "sample_rate", "boundary_status", "crop_ok"],
        "full_manifest",
    )
    assert acoustic_features["record_key"].is_unique
    manifest_tracking = full_manifest[
        ["record_key", "channels", "sample_rate", "boundary_status", "crop_ok"]
    ]
    frame = acoustic_features.merge(
        manifest_tracking,
        on="record_key",
        how="left",
        validate="one_to_one",
    )
    assert frame[["channels", "sample_rate", "boundary_status", "crop_ok"]].notna().all().all()

    training_duration = full_manifest.loc[
        full_manifest["split"] == "Training", "duration_sec"
    ].astype(float)
    duration_p01, duration_p99 = training_duration.quantile([0.01, 0.99])
    training_acoustic = frame.loc[
        (frame["split"] == "Training") & frame["extraction_ok"]
    ].copy()
    rms_p01, rms_p99 = weighted_quantile(
        training_acoustic["rms"], training_acoustic["sample_weight"], [0.01, 0.99]
    )
    silence_p99 = weighted_quantile(
        training_acoustic["silence_ratio"], training_acoustic["sample_weight"], [0.99]
    )[0]
    silence_low, silence_high = weighted_quantile(
        training_acoustic["silence_ratio"], training_acoustic["sample_weight"], [1 / 3, 2 / 3]
    )
    rms_low, rms_high = weighted_quantile(
        training_acoustic["rms"], training_acoustic["sample_weight"], [0.10, 0.90]
    )
    near_silence_threshold = 10 ** (-50.0 / 20)
    expected_sample_rate = int(full_manifest["sample_rate"].mode().iloc[0])
    expected_channels = int(full_manifest["channels"].mode().iloc[0])

    quality_status: list[str] = []
    quality_flags: list[list[str]] = []
    for row in frame.itertuples(index=False):
        invalid_flags: list[str] = []
        flags: list[str] = []
        if not bool(row.crop_ok):
            invalid_flags.append("invalid_crop")
        if not bool(row.extraction_ok):
            invalid_flags.append("extraction_failure")
        core_values = np.asarray([getattr(row, name) for name in CORE_ACOUSTIC_FEATURES], dtype=float)
        if bool(row.extraction_ok) and not np.isfinite(core_values).all():
            invalid_flags.append("non_finite_core_feature")
        if row.boundary_status in {"start_after_wav", "critical_overrun"}:
            invalid_flags.append("audio_length_mismatch")
        flags.extend(invalid_flags)
        if np.isfinite(row.duration_sec) and row.duration_sec <= duration_p01:
            flags.append("very_short_duration")
        if np.isfinite(row.duration_sec) and row.duration_sec >= duration_p99:
            flags.append("very_long_duration")
        if np.isfinite(row.rms) and row.rms <= rms_p01:
            flags.append("low_rms")
        if np.isfinite(row.rms) and row.rms >= rms_p99:
            flags.append("high_rms")
        if np.isfinite(row.rms) and row.rms <= near_silence_threshold:
            flags.append("near_silence_absolute")
        if np.isfinite(row.silence_ratio) and row.silence_ratio >= silence_p99:
            flags.append("high_silence_ratio")
        if np.isfinite(row.clipping_ratio) and row.clipping_ratio > 0:
            flags.append("clipping_detected")
        if np.isfinite(row.sample_rate) and int(row.sample_rate) != expected_sample_rate:
            flags.append("sample_rate_anomaly")
        if np.isfinite(row.channels) and int(row.channels) != expected_channels:
            flags.append("channel_anomaly")
        quality_flags.append(flags)
        quality_status.append("invalid" if invalid_flags else "suspicious" if flags else "normal")

    frame["quality_flags"] = quality_flags
    frame["quality_status"] = quality_status
    silence_edges, silence_labels = adaptive_three_level_bins(
        silence_low, silence_high, ["low", "medium", "high"], "silence"
    )
    rms_edges, rms_labels = adaptive_three_level_bins(
        rms_low, rms_high, ["low_rms", "normal_rms", "high_rms"], "RMS"
    )
    frame["silence_slice"] = pd.cut(
        frame["silence_ratio"], bins=silence_edges, labels=silence_labels, include_lowest=True
    )
    frame["rms_slice"] = pd.cut(
        frame["rms"], bins=rms_edges, labels=rms_labels, include_lowest=True
    )
    thresholds = pd.DataFrame(
        [
            ("very_short_duration", duration_p01, "Training P01"),
            ("very_long_duration", duration_p99, "Training P99"),
            ("low_rms", rms_p01, "weighted Training acoustic P01"),
            ("high_rms", rms_p99, "weighted Training acoustic P99"),
            ("high_silence_ratio", silence_p99, "weighted Training acoustic P99"),
            ("silence_slice_low", silence_low, "weighted Training acoustic P33"),
            ("silence_slice_high", silence_high, "weighted Training acoustic P67"),
            ("rms_slice_low", rms_low, "weighted Training acoustic P10"),
            ("rms_slice_high", rms_high, "weighted Training acoustic P90"),
        ],
        columns=["threshold", "value", "basis"],
    )
    return frame, thresholds


In [23]:
LISTENING_NOTE_COLUMNS = [
    "loudness", "noise", "silence", "overlap", "clarity", "clipping",
    "channel_feel", "speech_style", "emotion", "guessed_role", "memo",
]


def select_listening_candidates(
    ledger: pd.DataFrame,
    *,
    per_group: int,
    max_per_call: int,
    high_confidence_threshold: float,
    low_confidence_threshold: float,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Select four balanced groups while enforcing a global per-call cap."""
    require_columns(
        ledger,
        ["record_key", "call_id", "utt_idx", "audio_path", "startAt", "endAt", "duration",
         "y_true", "y_pred", "p0", "p1", "confidence", "correct", "error_type", "overlap_status"],
        "prediction ledger",
    )
    assert per_group > 0 and max_per_call > 0
    definitions = [
        ("high_confidence_0_to_1", (ledger["y_true"] == 0) & (ledger["y_pred"] == 1)
         & (ledger["confidence"] >= high_confidence_threshold), False),
        ("high_confidence_1_to_0", (ledger["y_true"] == 1) & (ledger["y_pred"] == 0)
         & (ledger["confidence"] >= high_confidence_threshold), False),
        ("low_confidence_correct_0", (ledger["y_true"] == 0) & ledger["correct"]
         & (ledger["confidence"] <= low_confidence_threshold), True),
        ("low_confidence_correct_1", (ledger["y_true"] == 1) & ledger["correct"]
         & (ledger["confidence"] <= low_confidence_threshold), True),
    ]
    call_usage: Counter[str] = Counter()
    selected_parts: list[pd.DataFrame] = []
    for group_name, mask, ascending in definitions:
        candidates = ledger.loc[mask].sort_values(
            ["confidence", "record_key"],
            ascending=[ascending, True],
            kind="stable",
        )
        selected_indices: list[int] = []
        for position, row in candidates.iterrows():
            call_id = str(row["call_id"])
            if call_usage[call_id] >= max_per_call:
                continue
            selected_indices.append(position)
            call_usage[call_id] += 1
            if len(selected_indices) == per_group:
                break
        if len(selected_indices) != per_group:
            raise RuntimeError(
                f"{group_name}: requested {per_group}, selected {len(selected_indices)} "
                f"under max_per_call={max_per_call}. Do not relax silently."
            )
        part = ledger.loc[selected_indices].copy()
        part["candidate_group"] = group_name
        selected_parts.append(part)

    hidden_key = pd.concat(selected_parts, ignore_index=True)
    hidden_key.insert(0, "sample_id", [f"BLIND-{index:02d}" for index in range(1, len(hidden_key) + 1)])
    blind_columns = [
        "sample_id", "record_key", "call_id", "utt_idx", "audio_path",
        "startAt", "endAt", "duration", "overlap_status",
    ]
    blind_candidates = hidden_key[blind_columns].copy()
    forbidden = {"y_true", "y_pred", "p0", "p1", "confidence", "correct", "error_type", "candidate_group"}
    assert forbidden.isdisjoint(blind_candidates.columns)
    notes = pd.DataFrame({"sample_id": blind_candidates["sample_id"]})
    for column in LISTENING_NOTE_COLUMNS:
        notes[column] = ""
    return hidden_key, blind_candidates, notes


def validate_and_reveal_listening_key(
    notes: pd.DataFrame,
    hidden_key: pd.DataFrame,
    *,
    reveal_enabled: bool,
) -> pd.DataFrame:
    if not reveal_enabled:
        raise RuntimeError("Set REVEAL_LISTENING_ANSWERS=True only after completing every note field.")
    require_columns(notes, ["sample_id", *LISTENING_NOTE_COLUMNS], "listening notes")
    assert notes["sample_id"].is_unique
    expected_ids = hidden_key["sample_id"].tolist()
    assert notes["sample_id"].tolist() == expected_ids, "Listening sample order changed"
    incomplete = notes[LISTENING_NOTE_COLUMNS].apply(
        lambda column: column.astype(str).str.strip().eq("")
    )
    if incomplete.any().any():
        missing_cells = int(incomplete.sum().sum())
        raise RuntimeError(f"Listening notes are incomplete: {missing_cells} empty cells")
    return notes.merge(
        hidden_key[["sample_id", "candidate_group", "y_true", "y_pred", "confidence", "error_type"]],
        on="sample_id",
        how="left",
        validate="one_to_one",
    )


## Synthetic helper checks

이 cell은 합성 DataFrame만 사용한다. 실제 DCC JSON/WAV, MFCC cache, 모델을 읽지 않는다.
1:1 join 누락, class별 오류율 분모, half-open interval overlap, call 다양성 제한이 작동하는지 확인한다.


In [19]:
synthetic_manifest = pd.DataFrame(
    {
        "split": ["Validation"] * 3,
        "call_id": ["call-a", "call-a", "call-b"],
        "annotation_row": [0, 1, 0],
        "utterance_id": ["a0", "a1", "b0"],
        "startAt_ms": [0.0, 80.0, 0.0],
        "endAt_ms": [100.0, 180.0, 50.0],
        "speaker": [0, 1, 1],
        "duration_sec": [0.1, 0.1, 0.05],
        "audio_path": ["a.wav", "a.wav", "b.wav"],
        "json_path": ["a.json", "a.json", "b.json"],
        "crop_ok": [True, True, True],
        "boundary_status": ["within_wav"] * 3,
    }
)
synthetic_manifest["record_key"] = [
    make_record_key(split, call_id, row)
    for split, call_id, row in synthetic_manifest[["split", "call_id", "annotation_row"]].itertuples(index=False, name=None)
]
synthetic_bundle = {
    "stem": np.array(["call-a", "call-a", "call-b"]),
    "start_at": np.array([0.0, 80.0, 0.0]),
    "end_at": np.array([100.0, 180.0, 50.0]),
    "duration": np.array([100.0, 100.0, 50.0]),
    "y": np.array([0, 1, 1]),
}
synthetic_join, synthetic_audit = join_baseline_cache_to_manifest(
    synthetic_bundle, synthetic_manifest, expected_rows=3
)
assert synthetic_audit.loc[0, "missing_join_rows"] == 0

synthetic_metric_frame = pd.DataFrame(
    {"y_true": [0, 0, 0, 1, 1], "y_pred": [0, 1, 1, 1, 0], "confidence": [0.8, 0.9, 0.7, 0.6, 0.55]}
)
synthetic_summary = summarize_binary_frame(synthetic_metric_frame).iloc[0]
assert np.isclose(synthetic_summary["fp_rate_within_true_0"], 2 / 3)
assert np.isclose(synthetic_summary["fn_rate_within_true_1"], 1 / 2)

synthetic_overlap = add_overlap_metadata(
    pd.DataFrame(
        {"record_key": ["a", "b", "c"], "call_id": ["x", "x", "x"],
         "startAt": [0.0, 50.0, 100.0], "endAt": [100.0, 80.0, 200.0]}
    )
)
assert synthetic_overlap["overlap_partner_count"].tolist() == [1, 1, 0]
print("SYNTHETIC_HELPER_CHECKS_PASS — no competition data/model was used")


SYNTHETIC_HELPER_CHECKS_PASS — no competition data/model was used


# Checkpoint 1 — Baseline Freeze

1. **무엇/왜:** 기존 MFCC40 + Logistic Regression을 이후 실험의 고정 비교점으로 재현한다.
2. **측정:** Training/Validation row 수, confusion matrix, Accuracy, speaker별 Recall.
3. **입력:** 저장된 MFCC cache의 `X/y/stem/start_at/end_at/duration`; `X`는 MFCC40뿐이다.
4. **코드:** source/cache 계약 → 명시적 실행 switch → Training-only scaler/model → exact gate 순서다.
5. **확인 출력:** source hash, cache metadata, shape, exact matrix, matrix에서 유도된 metric.
6. **의미:** matrix가 같아야 Accuracy/Recall도 같은 예측 결과에서 나온다.
7. **오해 위험:** 저장된 성능 숫자를 prediction으로 복사하지 않는다. 모델이 새로 만든 prediction을 비교한다.
8. **꼭 이해:** scaler는 Training에 `fit_transform`, Validation에는 `transform`만 수행한다.
9. **다음 판단:** row 수 또는 matrix가 다르면 `BASELINE_GATE_PASSED`가 생성되지 않고 즉시 중단한다.

원 baseline 계약: 공식 Training 29,200 calls / 889,358 utterances, 공식 Validation 3,640 calls /
111,947 utterances, 8 kHz, `n_mfcc=20`, `n_fft=512`, `hop_length=160`, MFCC mean+std 40차원,
waveform preprocessing은 baseline과 동일하게 PCM crop을 float32로 읽고, 2차원이면 channel mean으로 mono화한 뒤
resampling 없이 `librosa.feature.mfcc`를 적용한다. 발화별 MFCC time-axis mean 20개와 std 20개를 연결한다.
`LogisticRegression(solver="lbfgs", max_iter=1000, class_weight=None, random_state=42)`.


In [25]:
def baseline_code_sha256(path: Path) -> str:
    notebook = json.loads(path.read_text(encoding="utf-8"))
    code_sources = [
        "".join(cell.get("source", []))
        for cell in notebook["cells"]
        if cell.get("cell_type") == "code"
    ]
    payload = "\n\n# %%\n\n".join(code_sources).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


assert BASELINE_NOTEBOOK_PATH.is_file(), BASELINE_NOTEBOOK_PATH
actual_source_hash = baseline_code_sha256(BASELINE_NOTEBOOK_PATH)
assert actual_source_hash == EXPECTED_BASELINE_CODE_SHA256, (
    "Baseline code changed; inspect the diff and redefine the freeze before proceeding",
    actual_source_hash,
    EXPECTED_BASELINE_CODE_SHA256,
)
assert len(MODEL_FEATURE_NAMES) == FEATURE_DIMENSION
assert not (set(MODEL_FEATURE_NAMES) & FORBIDDEN_MODEL_FIELDS)

cache_contract_rows = []
for split in ["Training", "Validation"]:
    tag = split.lower()
    metadata_path = BASELINE_CACHE_ROOT / f"{tag}_mfcc20_nfft512_hop160_v1.json"
    npz_path = BASELINE_CACHE_ROOT / f"{tag}_mfcc20_nfft512_hop160_v1.npz"
    assert metadata_path.is_file(), metadata_path
    assert npz_path.is_file(), npz_path
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    expected_metadata = {
        "cache_schema_version": "mission2_mfcc_mean_std_v1",
        "source_split": split,
        "source_call_count": EXPECTED_CALL_COUNTS[split],
        "mfcc_config": MFCC_CONFIG,
        "feature_dimension": FEATURE_DIMENSION,
        "utterance_count": EXPECTED_UTTERANCE_COUNTS[split],
    }
    assert metadata == expected_metadata, (metadata_path, metadata, expected_metadata)
    cache_contract_rows.append({"split": split, **metadata, "npz_path": str(npz_path)})

cache_contract = pd.DataFrame(cache_contract_rows)
display(cache_contract)
print("CHECKPOINT 1A PASS: baseline source and cache metadata match the frozen contract.")


,split,cache_schema_version,feature_dimension,mfcc_config,source_call_count,source_split,utterance_count,npz_path
0,Training,mission2_mfcc_mean_std_v1,40,"{'hop_length': 160, 'n_fft': 512, 'n_mfcc': 20...",29200,Training,889358,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...
1,Validation,mission2_mfcc_mean_std_v1,40,"{'hop_length': 160, 'n_fft': 512, 'n_mfcc': 20...",3640,Validation,111947,C:\Users\YW\Study\02_Projects\DCC_2026\prelim\...


CHECKPOINT 1A PASS: baseline source and cache metadata match the frozen contract.


- Training: **29,200 calls / 889,358 utterances**
- Validation: **3,640 calls / 111,947 utterances**
- MFCC feature: **40차원**
- MFCC 설정과 cache 정보가 기존 baseline 설정과 모두 일치했다.

기존 baseline을 다시 재현하는 데 사용할 데이터와 feature가 제대로 준비되어 있음을 확인했다.

아직 모델 성능을 다시 계산한 것은 아니므로, **Accuracy 80.91%가 재현됐다고 볼 수는 없다.**

→ 다음 단계에서 Logistic Regression을 다시 학습하고 기존 성능이 그대로 나오는지 확인한다.

### 실제 baseline reproduction — 사용자 실행 전용

아래 cell부터 실제 cache를 메모리에 올리고 Logistic Regression을 다시 학습한다.
`ea-config`에서 `RUN_BASELINE_REPRODUCTION=True`가 기본값이므로 Run All에서 실행될 수 있다.
cache가 없거나 계약이 다르면 Raw feature extraction을 자동 실행하지 않고 멈춘다.


In [7]:
if not RUN_BASELINE_REPRODUCTION:
    raise RuntimeError(
        "STOP: review Checkpoint 1A, then set RUN_BASELINE_REPRODUCTION=True in ea-config."
    )

def load_frozen_mfcc_cache(split: str) -> dict[str, np.ndarray]:
    path = BASELINE_CACHE_ROOT / f"{split.lower()}_mfcc20_nfft512_hop160_v1.npz"
    assert path.is_file(), f"Missing frozen cache; do not auto-extract here: {path}"
    with np.load(path, allow_pickle=False) as cache:
        bundle = {name: cache[name] for name in cache.files}
    assert set(bundle) == {"X", "y", "stem", "start_at", "end_at", "duration"}
    expected_rows = EXPECTED_UTTERANCE_COUNTS[split]
    assert bundle["X"].shape == (expected_rows, FEATURE_DIMENSION)
    assert bundle["y"].shape == (expected_rows,)
    assert all(len(bundle[name]) == expected_rows for name in ["stem", "start_at", "end_at", "duration"])
    assert set(np.unique(bundle["y"])) == {0, 1}
    assert np.isfinite(bundle["X"]).all()
    return bundle

full_train = load_frozen_mfcc_cache("Training")
full_validation = load_frozen_mfcc_cache("Validation")
assert len(np.unique(full_train["stem"])) == EXPECTED_CALL_COUNTS["Training"]
assert len(np.unique(full_validation["stem"])) == EXPECTED_CALL_COUNTS["Validation"]
assert not (set(full_train["stem"]) & set(full_validation["stem"]))
print("Training X/y:", full_train["X"].shape, full_train["y"].shape)
print("Validation X/y:", full_validation["X"].shape, full_validation["y"].shape)


Training X/y: (889358, 40) (889358,)
Validation X/y: (111947, 40) (111947,)


In [8]:
# X contains only the 40 named MFCC statistics. Tracking metadata remains outside X.
assert full_train["X"].shape[1] == len(MODEL_FEATURE_NAMES) == 40
baseline_scaler = StandardScaler()
X_train_scaled = baseline_scaler.fit_transform(full_train["X"])      # fit: Training only
X_validation_scaled = baseline_scaler.transform(full_validation["X"])  # transform: Validation only

baseline_model = LogisticRegression(
    solver="lbfgs",
    max_iter=1_000,
    class_weight=None,
    random_state=42,
)
baseline_model.fit(X_train_scaled, full_train["y"])
validation_predictions = baseline_model.predict(X_validation_scaled)
validation_probabilities = baseline_model.predict_proba(X_validation_scaled)
np.testing.assert_array_equal(baseline_model.classes_, np.array([0, 1]))
assert validation_predictions.shape == full_validation["y"].shape
assert validation_probabilities.shape == (EXPECTED_UTTERANCE_COUNTS["Validation"], 2)


In [11]:
observed_matrix, baseline_metric_table = evaluate_baseline_gate(
    full_validation["y"],
    validation_predictions,
    expected_rows=EXPECTED_UTTERANCE_COUNTS["Validation"],
    reference_matrix=REFERENCE_CONFUSION_MATRIX,
)
display(pd.DataFrame(observed_matrix, index=["true_0", "true_1"], columns=["pred_0", "pred_1"]))
display(baseline_metric_table)
BASELINE_GATE_PASSED = True
print("CHECKPOINT 1 PASS: exact row count and confusion matrix match; Accuracy/Recall were derived from that matrix.")


,pred_0,pred_1
true_0,41190,12655
true_1,8715,49387


,sample_count,correct_count,accuracy,speaker_0_recall,speaker_1_recall,fp_0_to_1,fn_1_to_0,fp_rate_within_true_0,fn_rate_within_true_1,support_0,support_1
0,111947,90577,0.809106,0.764974,0.850005,12655,8715,0.235026,0.149995,53845,58102


CHECKPOINT 1 PASS: exact row count and confusion matrix match; Accuracy/Recall were derived from that matrix.


### Checkpoint 1 결과 해석

- Validation 전체 111,947개 중 90,577개를 맞췄다.
- Accuracy는 80.91%다.
- 119대원(speaker=0) Recall: 76.50%
- 신고자(speaker=1) Recall: 85.00%
- 119대원을 신고자로 잘못 예측한 경우: 12,655개
- 신고자를 119대원으로 잘못 예측한 경우: 8,715개

기존 baseline의 confusion matrix와 성능이 정확히 재현됐다

또한 신고자보다 119대원을 더 자주 놓치는 경향이 확인된다.

→ 이제 이 baseline을 기준으로 오답이 어떤 상황에서 반복되는지 Error Analysis를 진행한다.

# Checkpoint 2 — Stable record_key + Prediction Ledger

1. **무엇/왜:** 모든 Validation prediction을 변경되지 않은 원본 JSON 발화 위치와 연결한다.
2. **측정:** `record_key` uniqueness, 원본 JSON↔EDA 및 baseline cache↔EDA 1:1 join.
3. **입력:** Validation JSON의 배열 순서와 허용 tracking 필드, EDA manifest, baseline prediction.
4. **코드:** JSON 배열을 `enumerate`해 `annotation_row`를 다시 만들고 두 번 독립 join한다.
5. **확인 출력:** 각 join 111,947행, missing 0, duplicate 0.
6. **의미:** `annotation_row`는 정렬로 추정한 값이 아니라 원본 JSON 배열의 zero-based 위치다.
7. **오해 위험:** JSON 파일 내용·배열 순서가 바뀌면 key도 바뀌며 gate가 멈춘다.
8. **꼭 이해:** `utt_idx`/timestamp/call ID는 추적 전용이며 `X`에 들어가지 않는다.
9. **다음 판단:** 어느 assertion이든 실패하면 ledger와 Error Analysis를 생성하지 않는다.


In [14]:
assert globals().get("BASELINE_GATE_PASSED", False), "Checkpoint 1 gate did not pass"
assert DATA_ROOT.is_dir(), f"Official data root not found: {DATA_ROOT}"
assert EDA_UTTERANCE_PATH.is_file(), (
    f"Missing Raw EDA cache: {EDA_UTTERANCE_PATH}. This notebook will not rerun Raw EDA."
)
full_eda_manifest = pd.read_parquet(EDA_UTTERANCE_PATH)
assert len(full_eda_manifest) == sum(EXPECTED_UTTERANCE_COUNTS.values())
assert full_eda_manifest["record_key"].is_unique
eda_validation = (
    full_eda_manifest.loc[full_eda_manifest["split"] == "Validation"]
    .copy()
    .sort_values(["call_id", "annotation_row"], kind="stable")
    .reset_index(drop=True)
)
assert len(eda_validation) == EXPECTED_UTTERANCE_COUNTS["Validation"]
assert eda_validation["record_key"].is_unique

original_validation_order = rebuild_original_json_order(
    eda_validation,
    DATA_ROOT,
    expected_rows=EXPECTED_UTTERANCE_COUNTS["Validation"],
)
_, source_join_audit = validate_eda_manifest_against_source(
    original_validation_order,
    eda_validation,
    expected_rows=EXPECTED_UTTERANCE_COUNTS["Validation"],
)
baseline_tracking_join, baseline_join_audit = join_baseline_cache_to_manifest(
    full_validation,
    eda_validation,
    expected_rows=EXPECTED_UTTERANCE_COUNTS["Validation"],
)
display(source_join_audit)
display(baseline_join_audit)
print("CHECKPOINT 2A PASS: annotation_row is reproduced from original JSON; both joins are exact 1:1.")


,expected_rows,source_rows,eda_rows,joined_rows,source_duplicate_keys,eda_duplicate_keys,missing_join_rows
0,111947,111947,111947,111947,0,0,0


,expected_rows,baseline_cache_rows,eda_manifest_rows,joined_rows,baseline_duplicate_keys,eda_duplicate_keys,duplicate_join_keys,missing_join_rows
0,111947,111947,111947,111947,0,0,0,0


CHECKPOINT 2A PASS: annotation_row is reproduced from original JSON; both joins are exact 1:1.


In [20]:
prediction_ledger = build_prediction_ledger(
    baseline_tracking_join,
    validation_predictions,
    validation_probabilities,
    baseline_model.classes_,
)
prediction_ledger = add_overlap_metadata(prediction_ledger)
assert len(prediction_ledger) == EXPECTED_UTTERANCE_COUNTS["Validation"]
assert prediction_ledger["record_key"].is_unique

ledger_columns = [
    "record_key", "call_id", "utterance_id", "utt_idx", "startAt", "endAt", "duration",
    "y_true", "y_pred", "p0", "p1", "confidence", "correct", "error_type",
    "overlap_status", "overlap_partner_count", "maximum_overlap_ratio",
    "audio_path", "json_path", "crop_ok", "boundary_status",
]
prediction_ledger = prediction_ledger[ledger_columns]
display(prediction_ledger.head(5))
print("Prediction ledger rows:", len(prediction_ledger))


,record_key,call_id,utterance_id,utt_idx,startAt,endAt,duration,y_true,y_pred,p0,...,confidence,correct,error_type,overlap_status,overlap_partner_count,maximum_overlap_ratio,audio_path,json_path,crop_ok,boundary_status
0,Validation|651e464d69a4f266f0626837_20220101|0,651e464d69a4f266f0626837_20220101,04d346b5,0,299.0,949.0,0.650,0,1,0.440959,...,0.559041,False,0_to_1,no_overlap,0,0.000000,Validation/1.원천데이터/VS_서울_구급/651e464d69a4f266f0...,Validation/2.라벨링데이터/VL_서울_구급/651e464d69a4f266f...,True,within_wav
1,Validation|651e464d69a4f266f0626837_20220101|1,651e464d69a4f266f0626837_20220101,6c283804,1,2256.0,3074.0,0.818,1,1,0.143522,...,0.856478,True,correct_1,no_overlap,0,0.000000,Validation/1.원천데이터/VS_서울_구급/651e464d69a4f266f0...,Validation/2.라벨링데이터/VL_서울_구급/651e464d69a4f266f...,True,within_wav
2,Validation|651e464d69a4f266f0626837_20220101|2,651e464d69a4f266f0626837_20220101,d90085e1,2,3760.0,4577.0,0.817,0,0,0.895311,...,0.895311,True,correct_0,overlap,1,0.358629,Validation/1.원천데이터/VS_서울_구급/651e464d69a4f266f0...,Validation/2.라벨링데이터/VL_서울_구급/651e464d69a4f266f...,True,within_wav
3,Validation|651e464d69a4f266f0626837_20220101|3,651e464d69a4f266f0626837_20220101,ad5372a5,3,4284.0,5315.0,1.031,1,1,0.436026,...,0.563974,True,correct_1,overlap,1,0.358629,Validation/1.원천데이터/VS_서울_구급/651e464d69a4f266f0...,Validation/2.라벨링데이터/VL_서울_구급/651e464d69a4f266f...,True,within_wav
4,Validation|651e464d69a4f266f0626837_20220101|4,651e464d69a4f266f0626837_20220101,77748b09,4,5859.0,6223.0,0.364,0,0,0.631586,...,0.631586,True,correct_0,no_overlap,0,0.000000,Validation/1.원천데이터/VS_서울_구급/651e464d69a4f266f0...,Validation/2.라벨링데이터/VL_서울_구급/651e464d69a4f266f...,True,within_wav


Prediction ledger rows: 111947


- Validation 전체 **111,947개 발화**에 대해 Prediction Ledger가 생성됐다.
- 각 발화에 원본 위치, 정답/예측, 확률, confidence, overlap 정보 등이 연결됐다.
- 따라서 이제 오답을 원본 발화까지 추적하면서 분석할 수 있다.

모델이 틀린 샘플이 어떤 발화인지 1:1로 연결할 수 있는 분석용 장부가 만들어졌다.

→ 다음 단계에서 missing, duplicate, key 중복이 없는지 최종 확인한 뒤 Error Analysis를 진행한다.

In [31]:
# Duplicate conflict is a Training-only audit, not part of the Validation prediction population.
assert EDA_DUPLICATE_PATH.is_file(), EDA_DUPLICATE_PATH
duplicate_groups = pd.read_parquet(EDA_DUPLICATE_PATH)
assert len(duplicate_groups) == 3
assert int(duplicate_groups["rows"].sum()) == 6
assert set(duplicate_groups["split"]) == {"Training"}
assert set(duplicate_groups["speaker_values"]) == {"0,1"}

duplicate_conflict_rows = full_eda_manifest.merge(
    duplicate_groups[["split", "call_id", "startAt_ms", "endAt_ms"]],
    on=["split", "call_id", "startAt_ms", "endAt_ms"],
    how="inner",
    validate="many_to_one",
)
assert len(duplicate_conflict_rows) == 6
assert duplicate_conflict_rows["record_key"].is_unique
assert set(duplicate_conflict_rows["split"]) == {"Training"}
for _, group in duplicate_conflict_rows.groupby(["call_id", "startAt_ms", "endAt_ms"]):
    assert len(group) == 2 and set(group["speaker"].astype(int)) == {0, 1}

prediction_ledger["duplicate_conflict"] = prediction_ledger["record_key"].isin(
    duplicate_conflict_rows["record_key"]
)
assert int(prediction_ledger["duplicate_conflict"].sum()) == 0
display(duplicate_conflict_rows[
    ["record_key", "split", "call_id", "annotation_row", "utterance_id", "startAt_ms", "endAt_ms", "speaker"]
].sort_values(["call_id", "annotation_row"]))
print("Training-only duplicate conflict: 3 groups / 6 individually keyed rows; Validation matches: 0")
PREDICTION_LEDGER_GATE_PASSED = True
print("CHECKPOINT 2 PASS: stable 111,947-row ledger created with zero missing/duplicate joins.")


,record_key,split,call_id,annotation_row,utterance_id,startAt_ms,endAt_ms,speaker
0,Training|651e4b1829d82455557a200b_20220502|23,Training,651e4b1829d82455557a200b_20220502,23,wavesurfer_82b9a29978,52917.0,53599.0,1
1,Training|651e4b1829d82455557a200b_20220502|24,Training,651e4b1829d82455557a200b_20220502,24,wavesurfer_gaqavvtcq6g,52917.0,53599.0,0
2,Training|651e4bc5df6bf0d825aa56fa_20220505|23,Training,651e4bc5df6bf0d825aa56fa_20220505,23,wavesurfer_iedgleldo1g,44148.0,44839.0,0
3,Training|651e4bc5df6bf0d825aa56fa_20220505|24,Training,651e4bc5df6bf0d825aa56fa_20220505,24,wavesurfer_5609q1dal0o,44148.0,44839.0,1
4,Training|651e4c5b48bae907503799a9_20220602|17,Training,651e4c5b48bae907503799a9_20220602,17,wavesurfer_kd7dt2fr3so,28548.0,29321.0,0
5,Training|651e4c5b48bae907503799a9_20220602|18,Training,651e4c5b48bae907503799a9_20220602,18,wavesurfer_14b5qqq0p5,28548.0,29321.0,1


Training-only duplicate conflict: 3 groups / 6 individually keyed rows; Validation matches: 0
CHECKPOINT 2 PASS: stable 111,947-row ledger created with zero missing/duplicate joins.


In [32]:
if SAVE_ANALYSIS_ARTIFACTS:
    artifact_root = REPO_ROOT / "tmp" / "mission2_error_analysis"
    artifact_root.mkdir(parents=True, exist_ok=True)
    ledger_path = artifact_root / "validation_prediction_ledger.parquet"
    prediction_ledger.to_parquet(ledger_path, index=False)
    print("Saved:", ledger_path)
else:
    print("SAVE_ANALYSIS_ARTIFACTS=False: ledger remains in memory only.")


SAVE_ANALYSIS_ARTIFACTS=False: ledger remains in memory only.


### Checkpoint 2 결과 해석

- Validation 전체 111,947개 발화가 Prediction Ledger에 정확히 연결됐다.
- missing / duplicate join은 0개였다.
- 따라서 각 예측을 원본 발화까지 안정적으로 추적할 수 있다.
- 동일 시간구간에 반대 speaker label이 붙은 충돌은 Training에서만 3그룹 / 6행 있었고, Validation에는 0개였다.

Validation Error Analysis에 사용할 예측-원본 연결에는 문제가 없다.

→ 이제 어떤 조건에서 오답이 많이 발생하는지 분석할 수 있다.

# Checkpoint 3 — Baseline Error Analysis

1. **무엇/왜:** 어느 방향·확신도·조건에서 오류가 반복되는지 측정한다.
2. **측정:** Global, confidence, duration, acoustic/quality, overlap별 count/Accuracy/Recall/FP/FN/confidence.
3. **입력:** gate를 통과한 ledger와 기존 Raw EDA cache만 사용한다.
4. **코드:** 모집단별 표를 별도로 만들며 sample count를 항상 포함한다.
5. **확인 출력:** Global 111,947행; duration 합계 111,947행; acoustic 전체 1,200/Validation 600행.
6. **의미:** FP는 0→1, FN은 1→0이며 rate 분모는 각 실제 class support다.
7. **오해 위험:** acoustic 600행은 stratified sample이며 전체 Validation 성능이 아니다.
8. **꼭 이해:** confidence threshold는 분석 후보 분류일 뿐 공식 prediction threshold를 바꾸지 않는다.
9. **다음 판단:** 표본 수·대안 설명과 함께 반복 패턴을 확인한 뒤에만 가설로 기록한다.


In [35]:
assert globals().get("PREDICTION_LEDGER_GATE_PASSED", False), "Checkpoint 2 gate did not pass"
if not RUN_ERROR_ANALYSIS:
    raise RuntimeError("STOP: set RUN_ERROR_ANALYSIS=True only after reviewing Checkpoint 2 audits.")

global_summary = summarize_binary_frame(prediction_ledger)
assert int(global_summary.loc[0, "sample_count"]) == EXPECTED_UTTERANCE_COUNTS["Validation"]
display(global_summary)

confidence_edges = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99, np.inf]
confidence_labels = ["0.50–0.60", "0.60–0.70", "0.70–0.80", "0.80–0.90", "0.90–0.95", "0.95–0.99", "0.99–1.00"]
prediction_ledger["confidence_bin"] = pd.cut(
    prediction_ledger["confidence"],
    bins=confidence_edges,
    labels=confidence_labels,
    include_lowest=True,
    right=False,
)
confidence_summary = summarize_binary_frame(prediction_ledger, "confidence_bin")
assert int(confidence_summary["sample_count"].sum()) == EXPECTED_UTTERANCE_COUNTS["Validation"]
display(confidence_summary)

high_confidence_errors = prediction_ledger.loc[
    (~prediction_ledger["correct"])
    & (prediction_ledger["confidence"] >= HIGH_CONFIDENCE_THRESHOLD)
].sort_values(["confidence", "record_key"], ascending=[False, True])
low_confidence_correct = prediction_ledger.loc[
    prediction_ledger["correct"]
    & (prediction_ledger["confidence"] <= LOW_CONFIDENCE_THRESHOLD)
].sort_values(["confidence", "record_key"], ascending=[True, True])
print(f"Analysis-only thresholds: high >= {HIGH_CONFIDENCE_THRESHOLD:.2f}; low <= {LOW_CONFIDENCE_THRESHOLD:.2f}")
print("High-confidence errors:", len(high_confidence_errors))
print("Low-confidence correct:", len(low_confidence_correct))
display(high_confidence_errors.head(30))
display(low_confidence_correct.head(30))


,slice,sample_count,accuracy,speaker_0_recall,speaker_1_recall,fp_0_to_1,fn_1_to_0,fp_rate_within_true_0,fn_rate_within_true_1,support_0,support_1,mean_confidence,mean_error_confidence,correct_count
0,all,111947,0.809106,0.764974,0.850005,12655,8715,0.235026,0.149995,53845,58102,0.802446,0.687204,90577


,slice,sample_count,accuracy,speaker_0_recall,speaker_1_recall,fp_0_to_1,fn_1_to_0,fp_rate_within_true_0,fn_rate_within_true_1,support_0,support_1,mean_confidence,mean_error_confidence,correct_count
0,0.50–0.60,14835,0.570745,0.439886,0.688292,3932,2436,0.560114,0.311708,7020,7815,0.554645,0.552328,8467
1,0.60–0.70,20153,0.650077,0.447562,0.817646,5041,2011,0.552438,0.182354,9125,11028,0.647555,0.645851,13101
2,0.70–0.80,14596,0.774322,0.787175,0.760617,1603,1691,0.212825,0.239383,7532,7064,0.750926,0.746556,11302
3,0.80–0.90,20736,0.876061,0.893087,0.857273,1163,1407,0.106913,0.142727,10878,9858,0.854113,0.848844,18166
4,0.90–0.95,15901,0.926546,0.937694,0.915049,503,665,0.062306,0.084951,8073,7828,0.926731,0.924780,14733
5,0.95–0.99,19344,0.958902,0.960087,0.957920,350,445,0.039913,0.042080,8769,10575,0.971695,0.969352,18549
6,0.99–1.00,6382,0.980727,0.974265,0.984748,63,60,0.025735,0.015252,2448,3934,0.994869,0.994554,6259


Analysis-only thresholds: high >= 0.90; low <= 0.60
High-confidence errors: 2086
Low-confidence correct: 8467


,record_key,call_id,utterance_id,utt_idx,startAt,endAt,duration,y_true,y_pred,p0,...,error_type,overlap_status,overlap_partner_count,maximum_overlap_ratio,audio_path,json_path,crop_ok,boundary_status,duplicate_conflict,confidence_bin
54074,Validation|651e4f3cbd89bfb121b46c39_20220807|27,651e4f3cbd89bfb121b46c39_20220807,wavesurfer_aa9jg8g0nr8,27,64602.0,65402.0,0.800,1,0,0.999645,...,1_to_0,no_overlap,0,0.000000,Validation/1.원천데이터/VS_서울_구급/651e4f3cbd89bfb121...,Validation/2.라벨링데이터/VL_서울_구급/651e4f3cbd89bfb12...,True,within_wav,False,0.99–1.00
63161,Validation|651e4fb51952d0ca070e6b4f_20220906|11,651e4fb51952d0ca070e6b4f_20220906,08873989,11,25710.0,26299.0,0.589,1,0,0.999410,...,1_to_0,overlap,1,0.086587,Validation/1.원천데이터/VS_서울_구급/651e4fb51952d0ca07...,Validation/2.라벨링데이터/VL_서울_구급/651e4fb51952d0ca0...,True,within_wav,False,0.99–1.00
74944,Validation|651e51057f04b45fc969e5a1_20221102|11,651e51057f04b45fc969e5a1_20221102,8a51ea12,11,31064.0,31729.0,0.665,1,0,0.999242,...,1_to_0,no_overlap,0,0.000000,Validation/1.원천데이터/VS_서울_구급/651e51057f04b45fc9...,Validation/2.라벨링데이터/VL_서울_구급/651e51057f04b45fc...,True,within_wav,False,0.99–1.00
63541,Validation|651e4fcea6a4bdae3d84c371_20220907|8,651e4fcea6a4bdae3d84c371_20220907,wavesurfer_4776578rb0g,8,15683.0,17093.0,1.410,0,1,0.000814,...,0_to_1,overlap,1,1.000000,Validation/1.원천데이터/VS_서울_구급/651e4fcea6a4bdae3d...,Validation/2.라벨링데이터/VL_서울_구급/651e4fcea6a4bdae3...,True,within_wav,False,0.99–1.00
23572,Validation|651e4b1829d82455557a1f20_20220502|7,651e4b1829d82455557a1f20_20220502,206a92a5,7,17747.0,18080.0,0.333,1,0,0.999118,...,1_to_0,no_overlap,0,0.000000,Validation/1.원천데이터/VS_서울_구급/651e4b1829d8245555...,Validation/2.라벨링데이터/VL_서울_구급/651e4b1829d824555...,True,within_wav,False,0.99–1.00
57487,Validation|651e4f6d06957f3443ecfbf7_20220902|33,651e4f6d06957f3443ecfbf7_20220902,wavesurfer_tna9iastndg,33,65977.0,66915.0,0.938,0,1,0.000907,...,0_to_1,overlap,1,1.000000,Validation/1.원천데이터/VS_서울_구급/651e4f6d06957f3443...,Validation/2.라벨링데이터/VL_서울_구급/651e4f6d06957f344...,True,within_wav,False,0.99–1.00
66863,Validation|651e4ffdc97a3f0ef3c6a0ab_20221001|18,651e4ffdc97a3f0ef3c6a0ab_20221001,wavesurfer_gie3nkjjdk,18,35393.0,35957.0,0.564,0,1,0.000956,...,0_to_1,overlap,1,1.000000,Validation/1.원천데이터/VS_서울_구급/651e4ffdc97a3f0ef3...,Validation/2.라벨링데이터/VL_서울_구급/651e4ffdc97a3f0ef...,True,within_wav,False,0.99–1.00
83568,Validation|651e5258be98814e96ca2dc7_20221205|8,651e5258be98814e96ca2dc7_20221205,wavesurfer_684kaguavb,8,27532.0,28177.0,0.645,1,0,0.998984,...,1_to_0,no_overlap,0,0.000000,Validation/1.원천데이터/VS_서울_구급/651e5258be98814e96...,Validation/2.라벨링데이터/VL_서울_구급/651e5258be98814e9...,True,within_wav,False,0.99–1.00
103972,Validation|651e5500835c8a556d7e29f6_20220404|34,651e5500835c8a556d7e29f6_20220404,wavesurfer_v40hoci42o8,34,75629.0,76311.0,0.682,0,1,0.001067,...,0_to_1,overlap,1,0.609971,Validation/1.원천데이터/VS_서울_구급/651e5500835c8a556d...,Validation/2.라벨링데이터/VL_서울_구급/651e5500835c8a556...,True,within_wav,False,0.99–1.00
99497,Validation|651e54b0918cf05edbac487f_20220306|6,651e54b0918cf05edbac487f_20220306,wavesurfer_rvqqjcu62c,6,28471.0,30057.0,1.586,0,1,0.001132,...,0_to_1,overlap,1,0.915511,Validation/1.원천데이터/VS_서울_구급/651e54b0918cf05edb...,Validation/2.라벨링데이터/VL_서울_구급/651e54b0918cf05ed...,True,within_wav,False,0.99–1.00


,record_key,call_id,utterance_id,utt_idx,startAt,endAt,duration,y_true,y_pred,p0,...,error_type,overlap_status,overlap_partner_count,maximum_overlap_ratio,audio_path,json_path,crop_ok,boundary_status,duplicate_conflict,confidence_bin
50449,Validation|651e4f2166d8da7dcba972d2_20220806|35,651e4f2166d8da7dcba972d2_20220806,9147f808,35,78253.0,78706.0,0.453,1,1,0.499984,...,correct_1,overlap,1,1.000000,Validation/1.원천데이터/VS_서울_구급/651e4f2166d8da7dcb...,Validation/2.라벨링데이터/VL_서울_구급/651e4f2166d8da7dc...,True,within_wav,False,0.50–0.60
32252,Validation|651e4bfaf12d88de38d4975e_20220507|18,651e4bfaf12d88de38d4975e_20220507,087e8409,18,48406.0,48960.0,0.554,0,0,0.500084,...,correct_0,no_overlap,0,0.000000,Validation/1.원천데이터/VS_서울_구급/651e4bfaf12d88de38...,Validation/2.라벨링데이터/VL_서울_구급/651e4bfaf12d88de3...,True,within_wav,False,0.50–0.60
98756,Validation|651e54b0918cf05edbac4768_20220306|14,651e54b0918cf05edbac4768_20220306,29b32eee,14,38038.0,40713.0,2.675,0,0,0.500151,...,correct_0,no_overlap,0,0.000000,Validation/1.원천데이터/VS_서울_구급/651e54b0918cf05edb...,Validation/2.라벨링데이터/VL_서울_구급/651e54b0918cf05ed...,True,within_wav,False,0.50–0.60
107186,Validation|651e551aaa666596ee75fe10_20220405|22,651e551aaa666596ee75fe10_20220405,wavesurfer_kvkbvkken58,22,40704.0,41095.0,0.391,0,0,0.500188,...,correct_0,overlap,1,0.483376,Validation/1.원천데이터/VS_서울_구급/651e551aaa666596ee...,Validation/2.라벨링데이터/VL_서울_구급/651e551aaa666596e...,True,within_wav,False,0.50–0.60
52698,Validation|651e4f3cbd89bfb121b468fd_20220807|11,651e4f3cbd89bfb121b468fd_20220807,55279e24,11,30942.0,31273.0,0.331,1,1,0.499809,...,correct_1,no_overlap,0,0.000000,Validation/1.원천데이터/VS_서울_구급/651e4f3cbd89bfb121...,Validation/2.라벨링데이터/VL_서울_구급/651e4f3cbd89bfb12...,True,within_wav,False,0.50–0.60
95009,Validation|651e54776f6c84c1ff30cda1_20220304|2,651e54776f6c84c1ff30cda1_20220304,wavesurfer_miqc5vfcop8,2,5437.0,5946.0,0.509,0,0,0.500197,...,correct_0,overlap,1,1.000000,Validation/1.원천데이터/VS_서울_구급/651e54776f6c84c1ff...,Validation/2.라벨링데이터/VL_서울_구급/651e54776f6c84c1f...,True,within_wav,False,0.50–0.60
43165,Validation|651e4e712e98ee7120968c0e_20220704|7,651e4e712e98ee7120968c0e_20220704,13deb8a3,7,17373.0,18421.0,1.048,0,0,0.500198,...,correct_0,no_overlap,0,0.000000,Validation/1.원천데이터/VS_서울_구급/651e4e712e98ee7120...,Validation/2.라벨링데이터/VL_서울_구급/651e4e712e98ee712...,True,within_wav,False,0.50–0.60
175,Validation|651e464d69a4f266f062685f_20220101|17,651e464d69a4f266f062685f_20220101,14d11a2d,17,39734.0,44713.0,4.979,0,0,0.500206,...,correct_0,overlap,2,1.000000,Validation/1.원천데이터/VS_서울_구급/651e464d69a4f266f0...,Validation/2.라벨링데이터/VL_서울_구급/651e464d69a4f266f...,True,within_wav,False,0.50–0.60
7570,Validation|651e49c0c74dffea21f0f5b6_20220204|24,651e49c0c74dffea21f0f5b6_20220204,a9b2e218,24,45917.0,49587.0,3.670,1,1,0.499775,...,correct_1,overlap,2,0.799728,Validation/1.원천데이터/VS_서울_구급/651e49c0c74dffea21...,Validation/2.라벨링데이터/VL_서울_구급/651e49c0c74dffea2...,True,within_wav,False,0.50–0.60
59231,Validation|651e4f853f2c336d2bb0a147_20220903|22,651e4f853f2c336d2bb0a147_20220903,wavesurfer_fmfuce17qgg,22,35953.0,38217.0,2.264,0,0,0.500227,...,correct_0,no_overlap,0,0.000000,Validation/1.원천데이터/VS_서울_구급/651e4f853f2c336d2b...,Validation/2.라벨링데이터/VL_서울_구급/651e4f853f2c336d2...,True,within_wav,False,0.50–0.60


### Confidence 분석 결과 해석

- confidence가 높을수록 Accuracy도 높아지는 경향이 확인됐다.
- 하지만 confidence 0.90 이상인데도 틀린 **고확신 오답이 2,086개** 존재했다.
- confidence 0.60 이하인데 맞은 **저확신 정답은 8,467개**였다.
- 특히 낮은 confidence 구간에서 speaker=0 Recall이 크게 낮았다.

모델은 불확실할 때 119대원을 신고자로 잘못 분류하는 경향이 있고,
일부 샘플에서는 잘못된 예측을 매우 강하게 확신한다.

→ 다음 단계에서 고확신 오답이 duration, overlap, 음질 등 특정 조건에 몰리는지 확인한다.

In [21]:
# Full-population duration slices use Training P05/P25/P75/P95, matching the Raw EDA definition.
training_duration_quantiles = full_eda_manifest.loc[
    full_eda_manifest["split"] == "Training", "duration_sec"
].quantile([0.05, 0.25, 0.75, 0.95])
duration_edges = [-np.inf, *training_duration_quantiles.tolist(), np.inf]
duration_labels = ["very_short", "short", "medium", "long", "very_long"]
# 중복 quantile에서는 폭이 0인 구간과 해당 label만 제거해 나머지 경계를 유지한다.
if len(set(duration_edges)) < len(duration_edges):
    nonempty_intervals = [
        (right, label)
        for left, right, label in zip(duration_edges[:-1], duration_edges[1:], duration_labels)
        if left < right
    ]
    duration_edges = [duration_edges[0], *[right for right, _ in nonempty_intervals]]
    duration_labels = [label for _, label in nonempty_intervals]
prediction_ledger["duration_slice"] = pd.cut(
    prediction_ledger["duration"],
    bins=duration_edges,
    labels=duration_labels,
    include_lowest=True,
)
duration_summary = summarize_binary_frame(prediction_ledger, "duration_slice")
assert int(duration_summary["sample_count"].sum()) == EXPECTED_UTTERANCE_COUNTS["Validation"]
print("SCOPE: duration = full official Validation, 111,947 rows")
display(training_duration_quantiles.rename("seconds").to_frame())
display(duration_summary)


SCOPE: duration = full official Validation, 111,947 rows


,seconds
0.05,0.430
0.25,0.846
0.75,2.550
0.95,5.418


,slice,sample_count,accuracy,speaker_0_recall,speaker_1_recall,fp_0_to_1,fn_1_to_0,fp_rate_within_true_0,fn_rate_within_true_1,support_0,support_1,mean_confidence,mean_error_confidence,correct_count
0,very_short,5786,0.790010,0.803683,0.776479,565,650,0.196317,0.223521,2878,2908,0.832178,0.723206,4571
1,short,22545,0.755068,0.717606,0.783480,2746,2776,0.282394,0.216520,9724,12821,0.808020,0.711796,17023
2,medium,55659,0.797787,0.750302,0.852667,7451,3804,0.249698,0.147333,29840,25819,0.793645,0.678703,44404
3,long,22384,0.871024,0.829115,0.903190,1661,1226,0.170885,0.096810,9720,12664,0.807531,0.664122,19497
4,very_long,5573,0.911897,0.862151,0.933419,232,259,0.137849,0.066581,1683,3890,0.816495,0.652139,5082


### Duration 분석 결과 해석

- 전체 Validation 발화 길이의 중앙 구간은 대략 **0.85~2.55초**였다.
- Accuracy는
  - very_short: **79.0%**
  - short: **75.5%**
  - medium: **79.8%**
  - long: **87.1%**
  - very_long: **91.2%**
- 특히 **short 구간에서 성능이 가장 낮았고**, 긴 발화에서는 성능이 크게 높아졌다.
- 대부분의 구간에서 speaker=0 Recall이 speaker=1보다 낮았다.
  
발화 길이와 모델 성능 사이에 관계가 있는 것으로 보인다.  
다만 길이가 길어서 성능이 좋아졌다고 바로 단정할 수는 없다.

→ 다음 실험에서 `duration`을 feature로 추가해 실제 성능 향상에 도움이 되는지 확인한다.

In [37]:
# Existing 1,200-row stratified cache only. No waveform or acoustic feature is re-extracted here.
assert EDA_ACOUSTIC_PATH.is_file(), (
    f"Missing Raw EDA acoustic cache: {EDA_ACOUSTIC_PATH}. This notebook will not rerun Raw EDA."
)
acoustic_features = pd.read_parquet(EDA_ACOUSTIC_PATH)
assert len(acoustic_features) == 1_200
assert acoustic_features["record_key"].is_unique
acoustic_slice_frame, acoustic_thresholds = build_acoustic_slice_frame(
    acoustic_features, full_eda_manifest
)
acoustic_validation = acoustic_slice_frame.loc[
    acoustic_slice_frame["split"] == "Validation"
].copy()
assert len(acoustic_validation) == 600

acoustic_ledger = acoustic_validation.merge(
    prediction_ledger,
    on="record_key",
    how="left",
    suffixes=("_eda", ""),
    indicator=True,
    validate="one_to_one",
)
assert len(acoustic_ledger) == 600
assert (acoustic_ledger["_merge"] == "both").all()
np.testing.assert_array_equal(
    acoustic_ledger["speaker"].to_numpy(dtype=np.int64),
    acoustic_ledger["y_true"].to_numpy(dtype=np.int64),
)
print("SCOPE: acoustic/silence/RMS/quality = 600 Validation rows from the existing 1,200-row stratified sample")
print("These tables are NOT full-Validation estimates; always read sample_count.")
acoustic_link_columns = [
    "record_key", "rms", "silence_ratio", "silence_slice", "rms_slice",
    "quality_status", "quality_flags",
]
prediction_ledger = prediction_ledger.merge(
    acoustic_validation[acoustic_link_columns],
    on="record_key",
    how="left",
    validate="one_to_one",
)
prediction_ledger["acoustic_sampled"] = prediction_ledger["quality_status"].notna()
assert int(prediction_ledger["acoustic_sampled"].sum()) == 600
print("Linked acoustic slice metadata to 600 ledger rows; all other rows remain explicitly unsampled.")
display(acoustic_thresholds)
silence_summary = summarize_binary_frame(acoustic_ledger, "silence_slice")
rms_summary = summarize_binary_frame(acoustic_ledger, "rms_slice")
quality_summary = summarize_binary_frame(acoustic_ledger, "quality_status")
display(silence_summary)
display(rms_summary)
display(quality_summary)


SCOPE: acoustic/silence/RMS/quality = 600 Validation rows from the existing 1,200-row stratified sample
These tables are NOT full-Validation estimates; always read sample_count.
Linked acoustic slice metadata to 600 ledger rows; all other rows remain explicitly unsampled.


,threshold,value,basis
0,very_short_duration,0.292000,Training P01
1,very_long_duration,7.977000,Training P99
2,low_rms,0.007696,weighted Training acoustic P01
3,high_rms,0.242341,weighted Training acoustic P99
4,high_silence_ratio,0.764606,weighted Training acoustic P99
5,silence_slice_low,0.092387,weighted Training acoustic P33
6,silence_slice_high,0.216964,weighted Training acoustic P67
7,rms_slice_low,0.026452,weighted Training acoustic P10
8,rms_slice_high,0.169910,weighted Training acoustic P90


,slice,sample_count,accuracy,speaker_0_recall,speaker_1_recall,fp_0_to_1,fn_1_to_0,fp_rate_within_true_0,fn_rate_within_true_1,support_0,support_1,mean_confidence,mean_error_confidence,correct_count
0,low,202,0.757426,0.626506,0.848739,31,18,0.373494,0.151261,83,119,0.742753,0.667392,153
1,medium,187,0.887701,0.893617,0.881720,10,11,0.106383,0.118280,94,93,0.834896,0.728175,166
2,high,211,0.872038,0.878049,0.863636,15,12,0.121951,0.136364,123,88,0.829838,0.674347,184


,slice,sample_count,accuracy,speaker_0_recall,speaker_1_recall,fp_0_to_1,fn_1_to_0,fp_rate_within_true_0,fn_rate_within_true_1,support_0,support_1,mean_confidence,mean_error_confidence,correct_count
0,low_rms,80,0.812500,0.824561,0.782609,10,5,0.175439,0.217391,57,23,0.811658,0.638836,65
1,normal_rms,468,0.844017,0.807018,0.879167,44,29,0.192982,0.120833,228,240,0.799137,0.689689,395
2,high_rms,52,0.826923,0.866667,0.810811,2,7,0.133333,0.189189,15,37,0.814017,0.696829,43


,slice,sample_count,accuracy,speaker_0_recall,speaker_1_recall,fp_0_to_1,fn_1_to_0,fp_rate_within_true_0,fn_rate_within_true_1,support_0,support_1,mean_confidence,mean_error_confidence,correct_count
0,normal,551,0.840290,0.809187,0.873134,54,34,0.190813,0.126866,283,268,0.798965,0.682197,463
1,suspicious,49,0.816327,0.882353,0.781250,2,7,0.117647,0.218750,17,32,0.837304,0.685330,40


### Acoustic / Quality 분석 결과 해석

- **Silence**
  - low: Accuracy **75.7%**
  - medium: **88.8%**
  - high: **87.2%**
  - 특히 low-silence에서 speaker=0 Recall이 **62.7%**로 낮았다.

- **RMS**
  - low / normal / high Accuracy가 약 **81~84%**로 큰 차이는 없었다.
  - RMS만으로 뚜렷한 오류 패턴이 있다고 보기는 어렵다.

- **Quality**
  - normal: **84.0%**
  - suspicious: **81.6%**
  - suspicious가 조금 낮지만 표본이 **49개**뿐이라 확정하기 어렵다.
 
현재 표본에서는 silence가 오류와 관련될 가능성이 가장 눈에 띈다.  
반면 RMS와 suspicious 여부는 아직 강한 차이가 확인되지 않았다.

→ 단, 전체 Validation 결과가 아니므로 가설 수준으로만 두고 이후 실험에서 검증한다.

In [38]:
overlap_summary = summarize_binary_frame(prediction_ledger, "overlap_status")
assert int(overlap_summary["sample_count"].sum()) == EXPECTED_UTTERANCE_COUNTS["Validation"]
print("SCOPE: overlap = analysis/tracking metadata on full Validation; never a model feature")
display(overlap_summary)
print("CHECKPOINT 3 COMPLETE: results are descriptive observations, not automatic causes.")


SCOPE: overlap = analysis/tracking metadata on full Validation; never a model feature


,slice,sample_count,accuracy,speaker_0_recall,speaker_1_recall,fp_0_to_1,fn_1_to_0,fp_rate_within_true_0,fn_rate_within_true_1,support_0,support_1,mean_confidence,mean_error_confidence,correct_count
0,no_overlap,63084,0.842940,0.802918,0.878969,5890,4018,0.197082,0.121031,29886,33198,0.815662,0.677560,53176
1,overlap,48863,0.765426,0.717643,0.811396,6765,4697,0.282357,0.188604,23959,24904,0.785384,0.695541,37401


CHECKPOINT 3 COMPLETE: results are descriptive observations, not automatic causes.


### Overlap 분석 결과 해석

- overlap 없음: Accuracy **84.3%**
- overlap 있음: Accuracy **76.5%**
- overlap 구간의 speaker=0 Recall은 **71.8%**로 더 낮았다.
- speaker=0을 speaker=1로 잘못 분류하는 비율도 overlap 구간에서 더 높았다.

**해석:**  
겹쳐 말하는 발화에서 baseline 성능이 더 낮아지는 패턴이 확인됐다.

다만 overlap 자체가 직접적인 원인이라고 단정할 수는 없다.
발화 길이, 잡음, 음질 등 다른 요인이 함께 작용했을 가능성이 있다.

→ 이후 고확신 오답 청취와 다른 slice 분석을 함께 비교해 원인을 좁힌다.

# Checkpoint 4 — Representative blinded listening candidates

1. **무엇/왜:** 네 오류/경계 유형에서 실제 음성의 반복 패턴을 직접 듣는다.
2. **측정:** 고확신 0→1, 고확신 1→0, 저확신 정답 0, 저확신 정답 1.
3. **입력:** gate를 통과한 ledger와 해당 발화 crop만 사용한다. 주변 발화/text는 사용하지 않는다.
4. **코드:** global call cap을 적용해 후보를 고른 뒤 정답 관련 열을 숨긴다.
5. **확인 출력:** blind table과 call별 최대 선택 수.
6. **의미:** 청취 note는 가설 생성용이며 작은 표본의 성능 추정치가 아니다.
7. **오해 위험:** `record_key`와 시간은 재생 위치 추적용이다.
8. **꼭 이해:** reveal은 모든 note가 채워지고 별도 switch를 켜야 가능하다.
9. **다음 판단:** 반복 패턴은 counter-evidence와 함께 Checkpoint 5에 기록한다.


In [54]:
assert globals().get("PREDICTION_LEDGER_GATE_PASSED", False)
if not RUN_LISTENING_SELECTION:
    raise RuntimeError("STOP: set RUN_LISTENING_SELECTION=True after reviewing Error Analysis tables.")
listening_key, blind_candidates, listening_notes = select_listening_candidates(
    prediction_ledger,
    per_group=LISTENING_PER_GROUP,
    max_per_call=LISTENING_MAX_PER_CALL,
    high_confidence_threshold=HIGH_CONFIDENCE_THRESHOLD,
    low_confidence_threshold=LOW_CONFIDENCE_THRESHOLD,
)
assert listening_key["call_id"].value_counts().max() <= LISTENING_MAX_PER_CALL
display(blind_candidates)
display(listening_notes)
print("Hidden from blind table: y_true, y_pred, p0, p1, confidence, correct, error_type, candidate_group")


,sample_id,record_key,call_id,utt_idx,audio_path,startAt,endAt,duration,overlap_status
0,BLIND-01,Validation|651e4fcea6a4bdae3d84c371_20220907|8,651e4fcea6a4bdae3d84c371_20220907,8,Validation/1.원천데이터/VS_서울_구급/651e4fcea6a4bdae3d...,15683.0,17093.0,1.410,overlap
1,BLIND-02,Validation|651e4f6d06957f3443ecfbf7_20220902|33,651e4f6d06957f3443ecfbf7_20220902,33,Validation/1.원천데이터/VS_서울_구급/651e4f6d06957f3443...,65977.0,66915.0,0.938,overlap
2,BLIND-03,Validation|651e4ffdc97a3f0ef3c6a0ab_20221001|18,651e4ffdc97a3f0ef3c6a0ab_20221001,18,Validation/1.원천데이터/VS_서울_구급/651e4ffdc97a3f0ef3...,35393.0,35957.0,0.564,overlap
3,BLIND-04,Validation|651e4f3cbd89bfb121b46c39_20220807|27,651e4f3cbd89bfb121b46c39_20220807,27,Validation/1.원천데이터/VS_서울_구급/651e4f3cbd89bfb121...,64602.0,65402.0,0.800,no_overlap
4,BLIND-05,Validation|651e4fb51952d0ca070e6b4f_20220906|11,651e4fb51952d0ca070e6b4f_20220906,11,Validation/1.원천데이터/VS_서울_구급/651e4fb51952d0ca07...,25710.0,26299.0,0.589,overlap
5,BLIND-06,Validation|651e51057f04b45fc969e5a1_20221102|11,651e51057f04b45fc969e5a1_20221102,11,Validation/1.원천데이터/VS_서울_구급/651e51057f04b45fc9...,31064.0,31729.0,0.665,no_overlap
6,BLIND-07,Validation|651e4bfaf12d88de38d4975e_20220507|18,651e4bfaf12d88de38d4975e_20220507,18,Validation/1.원천데이터/VS_서울_구급/651e4bfaf12d88de38...,48406.0,48960.0,0.554,no_overlap
7,BLIND-08,Validation|651e54b0918cf05edbac4768_20220306|14,651e54b0918cf05edbac4768_20220306,14,Validation/1.원천데이터/VS_서울_구급/651e54b0918cf05edb...,38038.0,40713.0,2.675,no_overlap
8,BLIND-09,Validation|651e551aaa666596ee75fe10_20220405|22,651e551aaa666596ee75fe10_20220405,22,Validation/1.원천데이터/VS_서울_구급/651e551aaa666596ee...,40704.0,41095.0,0.391,overlap
9,BLIND-10,Validation|651e4f2166d8da7dcba972d2_20220806|35,651e4f2166d8da7dcba972d2_20220806,35,Validation/1.원천데이터/VS_서울_구급/651e4f2166d8da7dcb...,78253.0,78706.0,0.453,overlap


,sample_id,loudness,noise,silence,overlap,clarity,clipping,channel_feel,speech_style,emotion,guessed_role,memo
0,BLIND-01,,,,,,,,,,,
1,BLIND-02,,,,,,,,,,,
2,BLIND-03,,,,,,,,,,,
3,BLIND-04,,,,,,,,,,,
4,BLIND-05,,,,,,,,,,,
5,BLIND-06,,,,,,,,,,,
6,BLIND-07,,,,,,,,,,,
7,BLIND-08,,,,,,,,,,,
8,BLIND-09,,,,,,,,,,,
9,BLIND-10,,,,,,,,,,,


Hidden from blind table: y_true, y_pred, p0, p1, confidence, correct, error_type, candidate_group


In [ ]:
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
from dcc_data.audio import read_wav_frames, read_wav_info
from dcc_data.mission2 import milliseconds_to_frame

def play_blind_sample(sample_id: str) -> None:
    if not ENABLE_AUDIO_PLAYBACK:
        raise RuntimeError("Set ENABLE_AUDIO_PLAYBACK=True before playing a selected crop.")
    match = blind_candidates.loc[blind_candidates["sample_id"] == sample_id]
    assert len(match) == 1, sample_id
    row = match.iloc[0]
    wav_path = DATA_ROOT / row["audio_path"]
    info = read_wav_info(wav_path)
    start_frame = milliseconds_to_frame(float(row["startAt"]), info.sample_rate )
    end_frame = milliseconds_to_frame(float(row["endAt"]), info.sample_rate)
    segment = read_wav_frames(wav_path, start_frame, end_frame)
    print(sample_id, f"duration={float(row['duration']):.3f}s")
    display(Audio(segment.waveform.mean(axis=0), rate=segment.sample_rate))

# Example after enabling playback: play_blind_sample("BLIND-01")


In [ ]:
for sample_id in blind_candidates["sample_id"]:
    play_blind_sample(sample_id)

In [56]:
notes_data = {
    "BLIND-01": {
        "loudness": "normal",
        "noise": "mild",
        "silence": "none",
        "overlap": "none",
        "clarity": "unclear",
        "clipping": "none",
        "channel_feel": "muffled",
        "speech_style": "fast",
        "emotion": "anxious",
        "guessed_role": "unsure",
        "memo": "웅얼거려서 발화가 잘 들리지 않음",
    },
    "BLIND-02": {
        "loudness": "normal",
        "noise": "mild",
        "silence": "none",
        "overlap": "unsure",
        "clarity": "unclear",
        "clipping": "none",
        "channel_feel": "muffled",
        "speech_style": "fast",
        "emotion": "distressed",
        "guessed_role": "unsure",
        "memo": "웅얼거려서 발화가 잘 들리지 않음",
    },
    "BLIND-03": {
        "loudness": "normal",
        "noise": "strong",
        "silence": "much",
        "overlap": "unsure",
        "clarity": "unclear",
        "clipping": "suspected",
        "channel_feel": "muffled",
        "speech_style": "fast",
        "emotion": "other",
        "guessed_role": "unsure",
        "memo": "잡음과 무음이 많고 발화가 불명확함",
    },
    "BLIND-04": {
        "loudness": "low",
        "noise": "mild",
        "silence": "much",
        "overlap": "none",
        "clarity": "unclear",
        "clipping": "none",
        "channel_feel": "muffled",
        "speech_style": "other",
        "emotion": "neutral",
        "guessed_role": "unsure",
        "memo": "사람 목소리가 거의 들리지 않음",
    },
    "BLIND-05": {
        "loudness": "normal",
        "noise": "strong",
        "silence": "much",
        "overlap": "unsure",
        "clarity": "unclear",
        "clipping": "suspected",
        "channel_feel": "noisy",
        "speech_style": "other",
        "emotion": "other",
        "guessed_role": "unsure",
        "memo": "말하는 사람의 목소리가 뚜렷하게 들리지 않음",
    },
    "BLIND-06": {
        "loudness": "normal",
        "noise": "mild",
        "silence": "none",
        "overlap": "none",
        "clarity": "normal",
        "clipping": "none",
        "channel_feel": "noisy",
        "speech_style": "other",
        "emotion": "other",
        "guessed_role": "unsure",
        "memo": "사람의 한숨만 주로 들림",
    },
    "BLIND-07": {
        "loudness": "normal",
        "noise": "mild",
        "silence": "none",
        "overlap": "none",
        "clarity": "normal",
        "clipping": "suspected",
        "channel_feel": "noisy",
        "speech_style": "other",
        "emotion": "other",
        "guessed_role": "unsure",
        "memo": "너무 짧아서 판단하기 어려움",
    },
    "BLIND-08": {
        "loudness": "high",
        "noise": "none",
        "silence": "none",
        "overlap": "none",
        "clarity": "clear",
        "clipping": "none",
        "channel_feel": "clean",
        "speech_style": "calm",
        "emotion": "neutral",
        "guessed_role": "unsure",
        "memo": "발화가 매우 선명하고 차분하게 들림",
    },
    "BLIND-09": {
        "loudness": "low",
        "noise": "none",
        "silence": "much",
        "overlap": "none",
        "clarity": "clear",
        "clipping": "none",
        "channel_feel": "clean",
        "speech_style": "other",
        "emotion": "other",
        "guessed_role": "unsure",
        "memo": "삐 소리만 들리고 사람 발화는 거의 없음",
    },
    "BLIND-10": {
        "loudness": "low",
        "noise": "none",
        "silence": "much",
        "overlap": "none",
        "clarity": "clear",
        "clipping": "none",
        "channel_feel": "clean",
        "speech_style": "other",
        "emotion": "other",
        "guessed_role": "unsure",
        "memo": "삐 소리만 들리고 사람 발화는 거의 없음",
    },
    "BLIND-11": {
        "loudness": "normal",
        "noise": "none",
        "silence": "none",
        "overlap": "none",
        "clarity": "clear",
        "clipping": "none",
        "channel_feel": "clean",
        "speech_style": "calm",
        "emotion": "anxious",
        "guessed_role": "unsure",
        "memo": "매우 짧은 한 단어 발화만 들림",
    },
    "BLIND-12": {
        "loudness": "normal",
        "noise": "mild",
        "silence": "none",
        "overlap": "heard",
        "clarity": "unclear",
        "clipping": "none",
        "channel_feel": "noisy",
        "speech_style": "urgent",
        "emotion": "anxious",
        "guessed_role": "unsure",
        "memo": "두 사람 목소리가 모두 들림",
    },
}

for sample_id, values in notes_data.items():
    for column, value in values.items():
        listening_notes.loc[
            listening_notes["sample_id"] == sample_id, column
        ] = value

display(listening_notes)

# reveal 전에 빈칸이 없는지 확인
assert not listening_notes.drop(columns=["sample_id"]).eq("").any().any()
print("LISTENING_NOTES_COMPLETE")

,sample_id,loudness,noise,silence,overlap,clarity,clipping,channel_feel,speech_style,emotion,guessed_role,memo
0,BLIND-01,normal,mild,none,none,unclear,none,muffled,fast,anxious,unsure,웅얼거려서 발화가 잘 들리지 않음
1,BLIND-02,normal,mild,none,unsure,unclear,none,muffled,fast,distressed,unsure,웅얼거려서 발화가 잘 들리지 않음
2,BLIND-03,normal,strong,much,unsure,unclear,suspected,muffled,fast,other,unsure,잡음과 무음이 많고 발화가 불명확함
3,BLIND-04,low,mild,much,none,unclear,none,muffled,other,neutral,unsure,사람 목소리가 거의 들리지 않음
4,BLIND-05,normal,strong,much,unsure,unclear,suspected,noisy,other,other,unsure,말하는 사람의 목소리가 뚜렷하게 들리지 않음
5,BLIND-06,normal,mild,none,none,normal,none,noisy,other,other,unsure,사람의 한숨만 주로 들림
6,BLIND-07,normal,mild,none,none,normal,suspected,noisy,other,other,unsure,너무 짧아서 판단하기 어려움
7,BLIND-08,high,none,none,none,clear,none,clean,calm,neutral,unsure,발화가 매우 선명하고 차분하게 들림
8,BLIND-09,low,none,much,none,clear,none,clean,other,other,unsure,삐 소리만 들리고 사람 발화는 거의 없음
9,BLIND-10,low,none,much,none,clear,none,clean,other,other,unsure,삐 소리만 들리고 사람 발화는 거의 없음


LISTENING_NOTES_COMPLETE


In [ ]:
# Fill listening_notes in a separate editing cell or assign values programmatically.
# Use explicit values such as "none" rather than leaving a field blank.
if REVEAL_LISTENING_ANSWERS:
    revealed_listening = validate_and_reveal_listening_key(
        listening_notes,
        listening_key,
        reveal_enabled=REVEAL_LISTENING_ANSWERS,
    )
    display(revealed_listening)
else:
    print("Answers remain hidden. Complete every note field before enabling reveal.")


### Blind Listening 관찰

- 12개 모두 음성만 듣고 speaker 역할을 확신하기 어려웠다.
- 불명확한 발화, 많은 무음, 잡음, 짧은 발화, 겹침 등이 여러 샘플에서 관찰됐다.
- 사람 발화가 거의 없거나 비언어적 소리만 들리는 사례도 있었다.

**주의:**  
이 샘플들은 대표적인 오류/경계 사례로 선정된 소수 표본이므로 전체 Validation을 대표하지 않는다.

→ 정답과 모델 예측을 reveal한 뒤 어떤 특징이 실제 오류 유형과 반복적으로 연결되는지 확인한다.

### Blind Listening 결과 해석

- 고확신 오답 6개 대부분에서 웅얼거림, 잡음, 많은 무음, 비언어적 소리 등 역할 판단을 어렵게 만드는 음향 특징이 관찰됐다.
- 특히 일부 샘플은 사람 목소리가 거의 없는데도 모델이 반대 클래스를 **99.9% 수준으로 강하게 확신**했다.
- 저확신 정답에서는 짧은 발화나 삐 소리처럼 역할 정보가 거의 없는 사례가 있었고, confidence도 약 **0.50**에 가까웠다.
- 12개 모두 사람이 audio-only로 speaker 역할을 확신하기 어려웠다.

**해석:**  
baseline은 음성 정보가 부족하거나 불명확한 상황에서 어려움을 겪으며, 일부 경우에는 잘못된 음향 패턴을 과도하게 신뢰할 가능성이 있다.

→ 다음 단계에서 silence, quality, overlap 등의 통계 결과와 함께 비교해 실제 개선 가설을 정한다.

# Checkpoint 5 — Observation → Hypothesis log

1. **무엇/왜:** 표에서 보인 차이를 검증 가능한 다음 실험 질문으로 바꾼다.
2. **측정:** observation, repeated pattern, evidence, 반증, hypothesis, next experiment, decision metric.
3. **입력:** Checkpoint 3 표와 Checkpoint 4 청취 note.
4. **코드:** 원인을 자동 작성하지 않는 빈 기록 표다.
5. **확인 출력:** 사용자가 직접 채울 8개 빈 행.
6. **의미:** observation과 cause는 다르다.
7. **오해 위험:** 짧은 발화 성능 저하가 duration의 인과효과를 뜻하지 않는다.
8. **꼭 이해:** quality, overlap, class composition 등 대안 설명을 함께 기록한다.
9. **다음 판단:** 실제 결과를 본 뒤에만 단일 변경 실험을 결정한다.


In [60]:
HYPOTHESIS_COLUMNS = [
    "Observation",
    "Repeated pattern",
    "Possible cause",
    "Evidence",
    "Counter-evidence / Alternative explanation",
    "Improvement hypothesis",
    "Next experiment",
    "Decision metric",
]
hypothesis_log = pd.DataFrame(
    [{column: "" for column in HYPOTHESIS_COLUMNS} for _ in range(8)]
)
display(hypothesis_log)

if SAVE_ANALYSIS_ARTIFACTS:
    artifact_root = REPO_ROOT / "tmp" / "mission2_error_analysis"
    artifact_root.mkdir(parents=True, exist_ok=True)
    hypothesis_path = artifact_root / "observation_hypothesis_log.csv"
    hypothesis_log.to_csv(hypothesis_path, index=False, encoding="utf-8-sig")
    print("Saved blank/editable log:", hypothesis_path)


,Observation,Repeated pattern,Possible cause,Evidence,Counter-evidence / Alternative explanation,Improvement hypothesis,Next experiment,Decision metric
0,,,,,,,,
1,,,,,,,,
2,,,,,,,,
3,,,,,,,,
4,,,,,,,,
5,,,,,,,,
6,,,,,,,,
7,,,,,,,,


In [61]:
hypothesis_rows = [
    {
        "Observation": "speaker=0 성능이 speaker=1보다 낮다.",
        "Repeated pattern": "전체 Validation과 여러 slice에서 speaker=0 Recall이 더 낮게 나타난다.",
        "Possible cause": "현재 MFCC40 baseline이 speaker=0의 음향적 다양성을 충분히 구분하지 못할 가능성이 있다.",
        "Evidence": "전체 Validation Recall: speaker=0 76.50%, speaker=1 85.00%.",
        "Counter-evidence / Alternative explanation": "클래스별 난이도 차이 또는 데이터 구성 차이 때문일 수도 있어 원인은 아직 확정할 수 없다.",
        "Improvement hypothesis": "추가 acoustic feature가 speaker=0 분리에 도움이 될 수 있다.",
        "Next experiment": "이후 feature 실험마다 class별 Recall을 함께 비교한다.",
        "Decision metric": "Validation Accuracy 우선, speaker별 Recall 보조",
    },
    {
        "Observation": "낮은 confidence 구간에서 speaker=0을 특히 많이 놓친다.",
        "Repeated pattern": "불확실한 샘플에서 speaker=1 쪽으로 예측이 치우치는 경향이 있다.",
        "Possible cause": "결정경계 근처에서 두 class의 분리가 비대칭일 가능성이 있다.",
        "Evidence": "confidence 0.50~0.60 구간 Recall: speaker=0 43.99%, speaker=1 68.83%.",
        "Counter-evidence / Alternative explanation": "confidence 자체는 원인이 아니라 현재 모델 출력의 결과다.",
        "Improvement hypothesis": "추가 feature가 저확신 구간의 speaker=0 분리를 개선할 수 있다.",
        "Next experiment": "새 feature 실험 후 저확신 구간 Recall0 변화를 비교한다.",
        "Decision metric": "Validation Accuracy + low-confidence speaker=0 Recall",
    },
    {
        "Observation": "모델이 매우 확신하면서 틀리는 사례가 존재한다.",
        "Repeated pattern": "일부 어려운 발화를 반대 class로 높은 confidence로 예측한다.",
        "Possible cause": "MFCC가 일부 채널·무음·잡음·비언어적 음향 패턴을 speaker 신호로 과도하게 사용할 가능성이 있다.",
        "Evidence": "confidence >= 0.90인 고확신 오답 2,086개.",
        "Counter-evidence / Alternative explanation": "모든 고확신 오답이 동일한 원인이라고 볼 수는 없다.",
        "Improvement hypothesis": "추가 acoustic feature가 잘못된 강한 확신을 줄일 수 있다.",
        "Next experiment": "feature 추가 후 high-confidence error 수를 비교한다.",
        "Decision metric": "Validation Accuracy + high-confidence error count",
    },
    {
        "Observation": "짧은 발화에서 성능이 낮고 긴 발화에서 성능이 높다.",
        "Repeated pattern": "short 구간이 가장 낮고 long/very_long에서 Accuracy가 상승한다.",
        "Possible cause": "발화 길이가 현재 MFCC40에 없는 추가적인 분류 정보를 가질 가능성이 있다.",
        "Evidence": "Accuracy: very_short 79.0%, short 75.5%, medium 79.8%, long 87.1%, very_long 91.2%.",
        "Counter-evidence / Alternative explanation": "very_short가 short보다 높아 단순한 단조 관계는 아니며 duration이 직접 원인이라고 단정할 수 없다.",
        "Improvement hypothesis": "duration을 추가하면 baseline보다 Accuracy가 향상될 수 있다.",
        "Next experiment": "E01: MFCC40 + duration + Logistic Regression",
        "Decision metric": "동일 조건 official Validation Accuracy + class별 Recall",
    },
    {
        "Observation": "overlap 발화에서 성능이 낮다.",
        "Repeated pattern": "overlap 구간에서 두 class 모두 Recall이 감소한다.",
        "Possible cause": "두 화자 음성이 겹치면서 대상 화자의 acoustic pattern이 오염될 가능성이 있다.",
        "Evidence": "Accuracy: no_overlap 84.29%, overlap 76.54%.",
        "Counter-evidence / Alternative explanation": "overlap이 duration, noise 등 다른 조건과 함께 나타난 결과일 수 있어 인과관계는 확인되지 않았다.",
        "Improvement hypothesis": "새 feature가 overlap 상황에서도 더 안정적인 speaker 분리를 가능하게 할 수 있다.",
        "Next experiment": "overlap은 feature로 사용하지 않고 이후 모델의 overlap slice 성능을 비교한다.",
        "Decision metric": "전체 Validation Accuracy + overlap Accuracy",
    },
    {
        "Observation": "silence 구간에 따라 성능 차이가 관찰됐다.",
        "Repeated pattern": "표본에서 low-silence 구간의 Accuracy와 speaker=0 Recall이 낮았다.",
        "Possible cause": "silence 관련 특성이 현재 MFCC40 외에 추가적인 정보를 제공할 가능성이 있다.",
        "Evidence": "600개 acoustic sample Accuracy: low 75.7%, medium 88.8%, high 87.2%.",
        "Counter-evidence / Alternative explanation": "600개 표본 결과이므로 전체 Validation으로 일반화할 수 없다.",
        "Improvement hypothesis": "silence feature를 추가하면 일부 오류가 감소할 수 있다.",
        "Next experiment": "E02: silence feature 추가",
        "Decision metric": "Validation Accuracy + silence slice Accuracy",
    },
    {
        "Observation": "RMS 구간별 성능 차이는 크지 않았다.",
        "Repeated pattern": "low/normal/high RMS 모두 비슷한 수준의 Accuracy를 보였다.",
        "Possible cause": "RMS 단독으로는 speaker 분류에 추가 정보가 많지 않을 가능성이 있다.",
        "Evidence": "600개 acoustic sample에서 RMS slice Accuracy가 약 81~84% 범위.",
        "Counter-evidence / Alternative explanation": "표본 기반이며 다른 acoustic feature와 결합하면 효과가 나타날 수 있다.",
        "Improvement hypothesis": "RMS 단독 개선 효과는 작을 가능성이 있다.",
        "Next experiment": "E03: RMS 추가, 단 우선순위는 duration/silence보다 낮게 둔다.",
        "Decision metric": "Validation Accuracy 변화",
    },
    {
        "Observation": "청취한 어려운 샘플에서 불명확한 발화, 많은 무음, 잡음, 비언어적 소리가 반복됐다.",
        "Repeated pattern": "고확신 오답에서 웅얼거림, 잡음, 한숨, 사람 발화 부족 등이 여러 번 관찰됐다.",
        "Possible cause": "모델이 실제 speaker 정보보다 음질·무음·비언어적 acoustic cue에 반응하는 일부 사례가 있을 가능성이 있다.",
        "Evidence": "blind listening 12개 모두 guessed_role=unsure였고, 고확신 오답 6개 중 여러 샘플에서 해당 특징이 관찰됐다.",
        "Counter-evidence / Alternative explanation": "청취 표본은 12개뿐이며 대표성 있는 성능 표본이 아니다.",
        "Improvement hypothesis": "duration, silence, RMS, ZCR, spectral feature를 추가하면 일부 오류 유형이 감소할 수 있다.",
        "Next experiment": "E01~E05 수행 후 동일 listening/error 사례가 개선되는지 재확인한다.",
        "Decision metric": "Validation Accuracy + 해당 오류 사례 변화",
    },
]

hypothesis_log = pd.DataFrame(hypothesis_rows, columns=HYPOTHESIS_COLUMNS)

display(hypothesis_log)

,Observation,Repeated pattern,Possible cause,Evidence,Counter-evidence / Alternative explanation,Improvement hypothesis,Next experiment,Decision metric
0,speaker=0 성능이 speaker=1보다 낮다.,전체 Validation과 여러 slice에서 speaker=0 Recall이 더 ...,현재 MFCC40 baseline이 speaker=0의 음향적 다양성을 충분히 구분...,"전체 Validation Recall: speaker=0 76.50%, speake...",클래스별 난이도 차이 또는 데이터 구성 차이 때문일 수도 있어 원인은 아직 확정할 ...,추가 acoustic feature가 speaker=0 분리에 도움이 될 수 있다.,이후 feature 실험마다 class별 Recall을 함께 비교한다.,"Validation Accuracy 우선, speaker별 Recall 보조"
1,낮은 confidence 구간에서 speaker=0을 특히 많이 놓친다.,불확실한 샘플에서 speaker=1 쪽으로 예측이 치우치는 경향이 있다.,결정경계 근처에서 두 class의 분리가 비대칭일 가능성이 있다.,confidence 0.50~0.60 구간 Recall: speaker=0 43.9...,confidence 자체는 원인이 아니라 현재 모델 출력의 결과다.,추가 feature가 저확신 구간의 speaker=0 분리를 개선할 수 있다.,새 feature 실험 후 저확신 구간 Recall0 변화를 비교한다.,Validation Accuracy + low-confidence speaker=0...
2,모델이 매우 확신하면서 틀리는 사례가 존재한다.,일부 어려운 발화를 반대 class로 높은 confidence로 예측한다.,MFCC가 일부 채널·무음·잡음·비언어적 음향 패턴을 speaker 신호로 과도하게...,"confidence >= 0.90인 고확신 오답 2,086개.",모든 고확신 오답이 동일한 원인이라고 볼 수는 없다.,추가 acoustic feature가 잘못된 강한 확신을 줄일 수 있다.,feature 추가 후 high-confidence error 수를 비교한다.,Validation Accuracy + high-confidence error count
3,짧은 발화에서 성능이 낮고 긴 발화에서 성능이 높다.,short 구간이 가장 낮고 long/very_long에서 Accuracy가 상승한다.,발화 길이가 현재 MFCC40에 없는 추가적인 분류 정보를 가질 가능성이 있다.,"Accuracy: very_short 79.0%, short 75.5%, mediu...",very_short가 short보다 높아 단순한 단조 관계는 아니며 duration...,duration을 추가하면 baseline보다 Accuracy가 향상될 수 있다.,E01: MFCC40 + duration + Logistic Regression,동일 조건 official Validation Accuracy + class별 Re...
4,overlap 발화에서 성능이 낮다.,overlap 구간에서 두 class 모두 Recall이 감소한다.,두 화자 음성이 겹치면서 대상 화자의 acoustic pattern이 오염될 가능성...,"Accuracy: no_overlap 84.29%, overlap 76.54%.","overlap이 duration, noise 등 다른 조건과 함께 나타난 결과일 수...",새 feature가 overlap 상황에서도 더 안정적인 speaker 분리를 가능...,overlap은 feature로 사용하지 않고 이후 모델의 overlap slice...,전체 Validation Accuracy + overlap Accuracy
5,silence 구간에 따라 성능 차이가 관찰됐다.,표본에서 low-silence 구간의 Accuracy와 speaker=0 Recal...,silence 관련 특성이 현재 MFCC40 외에 추가적인 정보를 제공할 가능성이 있다.,"600개 acoustic sample Accuracy: low 75.7%, medi...",600개 표본 결과이므로 전체 Validation으로 일반화할 수 없다.,silence feature를 추가하면 일부 오류가 감소할 수 있다.,E02: silence feature 추가,Validation Accuracy + silence slice Accuracy
6,RMS 구간별 성능 차이는 크지 않았다.,low/normal/high RMS 모두 비슷한 수준의 Accuracy를 보였다.,RMS 단독으로는 speaker 분류에 추가 정보가 많지 않을 가능성이 있다.,600개 acoustic sample에서 RMS slice Accuracy가 약 8...,표본 기반이며 다른 acoustic feature와 결합하면 효과가 나타날 수 있다.,RMS 단독 개선 효과는 작을 가능성이 있다.,"E03: RMS 추가, 단 우선순위는 duration/silence보다 낮게 둔다.",Validation Accuracy 변화
7,"청취한 어려운 샘플에서 불명확한 발화, 많은 무음, 잡음, 비언어적 소리가 반복됐다.","고확신 오답에서 웅얼거림, 잡음, 한숨, 사람 발화 부족 등이 여러 번 관찰됐다.",모델이 실제 speaker 정보보다 음질·무음·비언어적 acoustic cue에 반...,"blind listening 12개 모두 guessed_role=unsure였고, ...",청취 표본은 12개뿐이며 대표성 있는 성능 표본이 아니다.,"duration, silence, RMS, ZCR, spectral feature를...",E01~E05 수행 후 동일 listening/error 사례가 개선되는지 재확인한다.,Validation Accuracy + 해당 오류 사례 변화


## 구현 종료 지점과 사용자 확인 순서

이 Notebook의 구현 범위는 baseline reproduction gate, stable prediction ledger, Global/confidence/
duration/acoustic-quality/overlap 분석, Training duplicate-conflict audit, blinded listening 후보,
observation→hypothesis 기록 구조까지다.

첫 실제 실행에서는 Checkpoint 1의 다음 값을 확인한다.

- Training/Validation shapes: `(889358, 40)`, `(111947, 40)`
- Validation row count: `111947`
- confusion matrix: `[[41190, 12655], [8715, 49387]]`
- Accuracy와 speaker별 Recall이 **그 matrix에서 다시 계산된 값**과 일치

즉시 중단 조건:

- baseline source/cache 계약 불일치
- row 수 또는 confusion matrix exact mismatch
- original JSON/EDA/baseline `record_key` 중복
- 어느 1:1 join에서든 joined rows ≠ 111,947 또는 missing rows > 0
- cache의 label/startAt/endAt/duration과 원본 manifest 불일치
- acoustic scope가 1,200 전체 / Validation 600이 아님

실제 Error Analysis 결과를 보기 전에는 feature 추가, threshold 변경, 모델 비교를 결정하지 않는다.
